In [1]:
!pip -q install pymupdf sentence-transformers pandas tqdm


In [6]:
from pathlib import Path
import requests
import fitz

In [7]:
PROJECT = Path("/content/rag_from_scratch")
DATA = PROJECT / "data"
RESULTS = PROJECT / "results"

for folder in [DATA, RESULTS]:
    folder.mkdir(parents=True, exist_ok=True)

In [5]:
pdf_path = DATA / "nist_ai_rmf.pdf"

# Alternative government-hosted PDF sources
urls = [
    "https://www.govinfo.gov/content/pkg/GOVPUB-C13-ef6882c6bd970bc733de394f469ea3f3/pdf/GOVPUB-C13-ef6882c6bd970bc733de394f469ea3f3.pdf",
    "https://rosap.ntl.bts.gov/view/dot/93262/dot_93262_DS1.pdf",
    "https://tsapps.nist.gov/publication/get_pdf.cfm?pub_id=936225"
]

headers = {
    "User-Agent": "Mozilla/5.0",
    "Accept": "application/pdf,*/*"
}

downloaded = False

for url in urls:
    try:
        print("Trying:", url)

        response = requests.get(
            url,
            headers=headers,
            timeout=45
        )
        response.raise_for_status()

        # Verify actual PDF content
        if not response.content.startswith(b"%PDF"):
            raise ValueError("Response is not a PDF")

        pdf_path.write_bytes(response.content)

        # Confirm the document is readable
        with fitz.open(pdf_path) as test_doc:
            if len(test_doc) < 30:
                raise ValueError("PDF has fewer than 30 pages")

        downloaded = True
        print("Download successful!")
        break

    except Exception as e:
        print("Failed:", str(e)[:200])

if not downloaded:
    raise RuntimeError(
        "Automatic download failed. Please manually upload "
        "the official NIST AI RMF PDF into Colab."
    )

# Extract pages
page_texts = []

with fitz.open(pdf_path) as doc:
    total_pages = len(doc)

    for page_number, page in enumerate(doc, start=1):
        text = page.get_text("text").strip()

        page_texts.append({
            "page": page_number,
            "text": text,
            "characters": len(text)
        })

print("\n========== DOCUMENT VALIDATION ==========")
print("Document:", pdf_path.name)
print("Pages:", total_pages)
print("File size (MB):", round(pdf_path.stat().st_size / 1e6, 2))
print("Nonempty pages:", sum(p["characters"] > 0 for p in page_texts))
print("Total characters:", sum(p["characters"] for p in page_texts))
print("First page preview:")
print(page_texts[0]["text"][:800])


Trying: https://www.govinfo.gov/content/pkg/GOVPUB-C13-ef6882c6bd970bc733de394f469ea3f3/pdf/GOVPUB-C13-ef6882c6bd970bc733de394f469ea3f3.pdf
Download successful!

========== DOCUMENT VALIDATION ==========
Document: nist_ai_rmf.pdf
Pages: 48
File size (MB): 2.05
Nonempty pages: 48
Total characters: 106427
First page preview:
NIST AI 100-1
Artificial Intelligence Risk Management
Framework (AI RMF 1.0)


# chunking pipeline

In [8]:
# CUSTOM TOKEN-BASED CHUNKING

import json
import re
from pathlib import Path
from transformers import AutoTokenizer

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def clean_text(text):
    """
    Normalize whitespace without changing source content.
    """
    text = text.replace("\x00", " ")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def chunk_document(pages, chunk_size, overlap):
    """
    Create overlapping token-based chunks.

    Each chunk stays within a single PDF page,
    preserving precise page-level citations.
    """
    assert 0 <= overlap < chunk_size

    chunks = []
    step = chunk_size - overlap

    for page in pages:
        page_number = page["page"]
        text = clean_text(page["text"])

        if not text:
            continue

        token_ids = tokenizer.encode(
            text,
            add_special_tokens=False
        )

        for start in range(0, len(token_ids), step):
            end = min(start + chunk_size, len(token_ids))
            piece = token_ids[start:end]

            if not piece:
                continue

            chunk_text = tokenizer.decode(
                piece,
                skip_special_tokens=True,
                clean_up_tokenization_spaces=False
            ).strip()

            if not chunk_text:
                continue

            chunks.append({
                "chunk_id": f"p{page_number}_t{start}",
                "page": page_number,
                "start_token": start,
                "end_token": end,
                "token_count": len(piece),
                "text": chunk_text
            })

            if end == len(token_ids):
                break

    return chunks


CONFIGS = {
    "small": {
        "chunk_size": 128,
        "overlap": 24
    },
    "large": {
        "chunk_size": 224,
        "overlap": 32
    }
}

all_chunks = {}

for name, config in CONFIGS.items():
    chunks = chunk_document(
        page_texts,
        chunk_size=config["chunk_size"],
        overlap=config["overlap"]
    )

    all_chunks[name] = chunks

    output_file = RESULTS / f"chunks_{name}.json"

    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(chunks, f, indent=2, ensure_ascii=False)

    token_counts = [c["token_count"] for c in chunks]

    print(f"\n========== {name.upper()} CHUNKS ==========")
    print("Chunk size:", config["chunk_size"])
    print("Overlap:", config["overlap"])
    print("Total chunks:", len(chunks))
    print("Average tokens:", round(sum(token_counts) / len(token_counts), 2))
    print("Minimum tokens:", min(token_counts))
    print("Maximum tokens:", max(token_counts))
    print("Pages covered:", len(set(c["page"] for c in chunks)))
    print("Saved to:", output_file)

print("\n========== SAMPLE CHUNK ==========")

sample = all_chunks["small"][0]

print("Chunk ID:", sample["chunk_id"])
print("PDF page:", sample["page"])
print("Token count:", sample["token_count"])
print("Text:\n")
print(sample["text"][:1000])


Loading tokenizer...


config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (624 > 512). Running this sequence through the model will result in indexing errors



========== SMALL CHUNKS ==========
Chunk size: 128
Overlap: 24
Total chunks: 212
Average tokens: 116.89
Minimum tokens: 19
Maximum tokens: 128
Pages covered: 48
Saved to: /content/rag_from_scratch/results/chunks_small.json

========== LARGE CHUNKS ==========
Chunk size: 224
Overlap: 32
Total chunks: 126
Average tokens: 185.24
Minimum tokens: 19
Maximum tokens: 224
Pages covered: 48
Saved to: /content/rag_from_scratch/results/chunks_large.json

========== SAMPLE CHUNK ==========
Chunk ID: p1_t0
PDF page: 1
Token count: 19
Text:

nist ai 100 - 1 artificial intelligence risk management framework ( ai rmf 1. 0 )


## Generate embeddings and build NumPy retrieval

In [9]:
# EMBEDDINGS + NUMPY COSINE SIMILARITY RETRIEVAL

import numpy as np
import pandas as pd
from pathlib import Path
from sentence_transformers import SentenceTransformer

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

print("Loading embedding model...")
model = SentenceTransformer(MODEL_NAME, device="cpu")
model.max_seq_length = 256

RESULTS = Path("/content/rag_from_scratch/results")
RESULTS.mkdir(parents=True, exist_ok=True)


def generate_embeddings(chunks, batch_size=32):
    """
    Convert document chunks into normalized embedding vectors.
    """
    texts = [chunk["text"] for chunk in chunks]

    embeddings = model.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    return embeddings.astype(np.float32)


print("\nGenerating embeddings for 128-token chunks...")
small_embeddings = generate_embeddings(all_chunks["small"])

print("\nGenerating embeddings for 224-token chunks...")
large_embeddings = generate_embeddings(all_chunks["large"])

np.save(RESULTS / "embeddings_small.npy", small_embeddings)
np.save(RESULTS / "embeddings_large.npy", large_embeddings)

print("\n========== EMBEDDING VALIDATION ==========")

print("Small embedding shape:", small_embeddings.shape)
print("Large embedding shape:", large_embeddings.shape)

assert small_embeddings.shape[0] == len(all_chunks["small"])
assert large_embeddings.shape[0] == len(all_chunks["large"])
assert small_embeddings.shape[1] == 384
assert large_embeddings.shape[1] == 384

print("All embedding dimensions validated.")


def retrieve(query, chunk_set="small", top_k=5):
    """
    Retrieve top-k document chunks using cosine similarity.

    Since document and query embeddings are L2-normalized,
    cosine similarity equals their dot product.
    """

    if chunk_set not in ("small", "large"):
        raise ValueError("chunk_set must be 'small' or 'large'")

    if chunk_set == "small":
        chunks = all_chunks["small"]
        embeddings = small_embeddings
    else:
        chunks = all_chunks["large"]
        embeddings = large_embeddings

    query_embedding = model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype(np.float32)[0]

    similarity_scores = embeddings @ query_embedding

    ranked_indices = np.argsort(-similarity_scores)[:top_k]

    results = []

    for rank, idx in enumerate(ranked_indices, start=1):
        chunk = chunks[int(idx)]

        results.append({
            "rank": rank,
            "chunk_id": chunk["chunk_id"],
            "page": chunk["page"],
            "score": float(similarity_scores[idx]),
            "text": chunk["text"]
        })

    return results


def display_retrieval(query, chunk_set="small", top_k=5):
    print("=" * 75)
    print("QUESTION:", query)
    print("CHUNK SET:", chunk_set)
    print("=" * 75)

    results = retrieve(query, chunk_set, top_k)

    for result in results:
        print(f"\nRank: {result['rank']}")
        print(f"PDF Page: {result['page']}")
        print(f"Similarity: {result['score']:.4f}")
        print(f"Chunk ID: {result['chunk_id']}")
        print("Passage:")
        print(result["text"][:900])
        print("-" * 75)

    return results


# INITIAL RETRIEVAL SMOKE TEST

test_query = "What are the four core functions of the AI Risk Management Framework?"

print("\n\nTESTING SMALL CHUNKS\n")
small_results = display_retrieval(
    test_query,
    chunk_set="small",
    top_k=5
)

print("\n\nTESTING LARGE CHUNKS\n")
large_results = display_retrieval(
    test_query,
    chunk_set="large",
    top_k=5
)


Loading embedding model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (4 outputs left out)

Batches:   0%|          | 0/7 [00:00<?, ?it/s]


Generating embeddings for 224-token chunks...


Batches:   0%|          | 0/4 [00:00<?, ?it/s]


========== EMBEDDING VALIDATION ==========
Small embedding shape: (212, 384)
Large embedding shape: (126, 384)
All embedding dimensions validated.


TESTING SMALL CHUNKS

QUESTION: What are the four core functions of the AI Risk Management Framework?
CHUNK SET: small

Rank: 1
PDF Page: 25
Similarity: 0.7885
Chunk ID: p25_t104
Passage:
##gories are subdivided into specific actions and outcomes. actions do not constitute a checklist, nor are they necessarily an ordered set of steps. fig. 5. functions organize ai risk management activities at their highest level to govern, map, measure, and manage ai risks. governance is designed to be a cross - cutting function to inform and be infused throughout the other three functions. risk management should be continuous, timely, and performed throughout the ai system lifecycle dimensions. ai rmf core functions should be carried out in a way that reflects diverse and multidisciplinary perspectives, potentially including the views of ai actors out -

## 15-question evaluation dataset

In [10]:
# PREPARE 15 GROUND-TRUTH ANNOTATIONS

import re
import json
import pandas as pd
from pathlib import Path

RESULTS = Path("/content/rag_from_scratch/results")
EVAL_DIR = Path("/content/rag_from_scratch/evaluation")
EVAL_DIR.mkdir(parents=True, exist_ok=True)

questions = [
    ("Q01", "What are the four core functions of the AI RMF?",
     "govern map measure manage"),
    ("Q02", "What is the purpose of the GOVERN function?",
     "govern function"),
    ("Q03", "What does the MAP function establish about an AI system?",
     "map function"),
    ("Q04", "What is the purpose of the MEASURE function?",
     "measure function"),
    ("Q05", "How does the MANAGE function address identified AI risks?",
     "manage function"),
    ("Q06", "What characteristics does NIST associate with trustworthy AI?",
     "trustworthy ai characteristics"),
    ("Q07", "What does valid and reliable mean in the context of AI systems?",
     "valid reliable"),
    ("Q08", "How does NIST describe safety in AI systems?",
     "safe safety"),
    ("Q09", "Why are security and resilience important for trustworthy AI?",
     "secure resilient"),
    ("Q10", "What is the difference between explainability and interpretability?",
     "explainability interpretability"),
    ("Q11", "How does the framework address privacy in AI systems?",
     "privacy enhanced"),
    ("Q12", "What role does harmful bias play in AI fairness?",
     "fair harmful bias"),
    ("Q13", "What are AI RMF profiles used for?",
     "profiles"),
    ("Q14", "How does the framework describe accountability and transparency?",
     "accountable transparent"),
    ("Q15", "Why is AI risk management considered a continuous process?",
     "continuous risk management")
]


def normalize_words(text):
    return set(re.findall(r"[a-z]+", text.lower()))


def find_candidate_pages(hint, pages, top_k=3):
    """
    Locate candidate evidence using keyword overlap
    directly against source pages.

    This is independent of the embedding retriever.
    """
    keywords = normalize_words(hint)
    ranked = []

    for page in pages:
        content = page["text"]
        words = normalize_words(content)

        matches = keywords.intersection(words)
        score = len(matches)

        if score:
            ranked.append({
                "page": page["page"],
                "score": score,
                "text": content
            })

    ranked.sort(key=lambda x: -x["score"])
    return ranked[:top_k]


records = []

for qid, question, hint in questions:
    candidates = find_candidate_pages(
        hint, page_texts, top_k=3
    )

    records.append({
        "question_id": qid,
        "question": question,
        "search_hint": hint,
        "candidate_pages": ",".join(
            str(c["page"]) for c in candidates
        ),
        "gold_page": "",
        "gold_passage": "",
        "verified": False
    })

    print(f"\n{qid}: {question}")
    print("Candidate PDF pages:",
          [c["page"] for c in candidates])

df = pd.DataFrame(records)

output_path = EVAL_DIR / "questions_15_draft.csv"
df.to_csv(output_path, index=False)

print("\n" + "=" * 60)
print("Total questions:", len(df))
print("Draft saved:", output_path)
print("Gold passages verified: 0 / 15")
print("=" * 60)



Q01: What are the four core functions of the AI RMF?
Candidate PDF pages: [4, 5, 8]

Q02: What is the purpose of the GOVERN function?
Candidate PDF pages: [4, 5, 25]

Q03: What does the MAP function establish about an AI system?
Candidate PDF pages: [4, 5, 25]

Q04: What is the purpose of the MEASURE function?
Candidate PDF pages: [4, 5, 9]

Q05: How does the MANAGE function address identified AI risks?
Candidate PDF pages: [4, 5, 6]

Q06: What characteristics does NIST associate with trustworthy AI?
Candidate PDF pages: [5, 7, 17]

Q07: What does valid and reliable mean in the context of AI systems?
Candidate PDF pages: [4, 5, 8]

Q08: How does NIST describe safety in AI systems?
Candidate PDF pages: [13, 19, 35]

Q09: Why are security and resilience important for trustworthy AI?
Candidate PDF pages: [4, 8, 17]

Q10: What is the difference between explainability and interpretability?
Candidate PDF pages: [21, 22, 11]

Q11: How does the framework address privacy in AI systems?
Candida

## Verify and save ground-truth passages

In [11]:

# ============================================================
# STEP 5: HUMAN-VERIFIED GROUND-TRUTH ANNOTATION
# ============================================================

import re
import json
import pandas as pd
from pathlib import Path

EVAL_DIR = Path("/content/rag_from_scratch/evaluation")
EVAL_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT = EVAL_DIR / "questions_15.csv"

# Preferred section ranges are search hints only.
# The actual evidence is selected by human review.
PREFERRED_PAGES = {
    "Q01": (25, 26),
    "Q02": (25, 29),
    "Q03": (28, 33),
    "Q04": (31, 36),
    "Q05": (35, 39),
    "Q06": (17, 24),
    "Q07": (17, 20),
    "Q08": (18, 21),
    "Q09": (19, 22),
    "Q10": (20, 23),
    "Q11": (21, 24),
    "Q12": (22, 25),
    "Q13": (38, 42),
    "Q14": (19, 23),
    "Q15": (23, 28)
}

STOPWORDS = {
    "what", "which", "where", "when", "why", "how",
    "does", "the", "and", "are", "for", "with",
    "from", "into", "this", "that", "about", "its",
    "their", "have", "has", "been", "being", "an",
    "of", "to", "in", "a", "is", "it"
}


def keywords(text):
    return set(re.findall(r"[a-z]+", text.lower())) - STOPWORDS


def make_windows(text, width=550, step=260):
    """
    Generate searchable source-text windows.
    Character positions refer to the normalized page text.
    """
    text_length = len(text)

    if not text_length:
        return []

    windows = []

    for start in range(0, text_length, step):
        end = min(start + width, text_length)

        # Avoid splitting in the middle of a word.
        if start > 0:
            while start < end and not text[start - 1].isspace():
                start += 1

        if end < text_length:
            while end > start and not text[end - 1].isspace():
                end -= 1

        if end <= start:
            continue

        passage = text[start:end].strip()

        if len(passage) >= 80:
            windows.append((start, end, passage))

        if end >= text_length:
            break

    return windows


def candidate_passages(qid, question, hint, top_k=6):
    query_terms = keywords(question + " " + hint)
    preferred_start, preferred_end = PREFERRED_PAGES[qid]

    candidates = []

    for page in page_texts:
        page_number = page["page"]

        # Exclude cover and front-matter pages from ranking,
        # while allowing them to be inspected manually.
        if page_number < 7:
            continue

        source_text = clean_text(page["text"])

        for start, end, passage in make_windows(source_text):
            passage_terms = keywords(passage)

            overlap = len(query_terms & passage_terms)

            if overlap == 0:
                continue

            coverage = overlap / max(len(query_terms), 1)
            score = coverage + 0.02 * overlap

            if preferred_start <= page_number <= preferred_end:
                score += 0.30

            # Prefer substantive explanatory prose.
            if len(passage) >= 300:
                score += 0.05

            candidates.append({
                "page": page_number,
                "start_char": start,
                "end_char": end,
                "text": passage,
                "score": round(score, 4)
            })

    candidates.sort(
        key=lambda x: (-x["score"], x["page"])
    )

    # Avoid showing many near-duplicate windows
    # from the same page.
    selected = []
    per_page = {}

    for candidate in candidates:
        p = candidate["page"]

        if per_page.get(p, 0) >= 2:
            continue

        selected.append(candidate)
        per_page[p] = per_page.get(p, 0) + 1

        if len(selected) >= top_k:
            break

    return selected


def inspect_page(page_number):
    page = next(
        (p for p in page_texts if p["page"] == page_number),
        None
    )

    if page is None:
        print("Invalid page number.")
        return

    print(f"\n========== PDF PAGE {page_number} ==========")
    print(page["text"][:12000])


# Resume annotations if the output file already exists.
if OUTPUT.exists():
    saved = pd.read_csv(OUTPUT).fillna("")
    annotations = saved.to_dict("records")
else:
    annotations = []

completed = {row["question_id"] for row in annotations}

print("Existing annotations:", len(completed))
print("Type a candidate number to select its passage.")
print("Type p25 to inspect PDF page 25.")
print("Type s to skip a question.")
print("Type q to save and exit.\n")

for qid, question, hint in questions:

    if qid in completed:
        continue

    print("\n" + "=" * 75)
    print(qid, question)
    print("=" * 75)

    candidates = candidate_passages(qid, question, hint)

    for number, candidate in enumerate(candidates, start=1):
        print(f"\n[{number}] PDF PAGE {candidate['page']}")
        print(candidate["text"])
        print("-" * 65)

    while True:
        selection = input(
            "\nChoose evidence [1-6], p<page>, s, or q: "
        ).strip().lower()

        if selection == "q":
            break

        if selection == "s":
            print("Skipped:", qid)
            break

        if selection.startswith("p") and selection[1:].isdigit():
            inspect_page(int(selection[1:]))
            continue

        if selection.isdigit():
            index = int(selection) - 1

            if 0 <= index < len(candidates):
                chosen = candidates[index]

                # Confirm that the evidence actually answers
                # the complete question.
                print("\nSELECTED SOURCE PASSAGE:")
                print(chosen["text"])

                confirm = input(
                    "\nDoes this passage fully support the answer? "
                    "[y/n]: "
                ).strip().lower()

                if confirm != "y":
                    print("Not accepted. Choose another passage.")
                    continue

                record = {
                    "question_id": qid,
                    "question": question,
                    "gold_page": chosen["page"],
                    "gold_passage": chosen["text"],
                    "start_char": chosen["start_char"],
                    "end_char": chosen["end_char"],
                    "verified": True
                }

                annotations.append(record)
                completed.add(qid)

                pd.DataFrame(annotations).to_csv(
                    OUTPUT, index=False
                )

                print("Saved:", qid)
                break

        print("Invalid selection. Try again.")

    if selection == "q":
        break


print("\n========== ANNOTATION STATUS ==========")
print("Verified:", len(completed), "/ 15")
print("Saved to:", OUTPUT)

if len(completed) == 15:
    print("All 15 questions have annotated passages.")
else:
    print("Run this cell again to continue unfinished questions.")


Existing annotations: 0
Type a candidate number to select its passage.
Type p25 to inspect PDF page 25.
Type s to skip a question.
Type q to save and exit.


Q01 What are the four core functions of the AI RMF?

[1] PDF PAGE 25
NIST AI 100-1
AI RMF 1.0
Part 2: Core and Profiles
5.
AI RMF Core
The AI RMF Core provides outcomes and actions that enable dialogue, understanding, and
activities to manage AI risks and responsibly develop trustworthy AI systems. As illus-
trated in Figure 5, the Core is composed of four functions: GOVERN, MAP, MEASURE,
and MANAGE. Each of these high-level functions is broken down into categories and sub-
categories. Categories and subcategories are subdivided into specific actions and outcomes.
Actions do not constitute a checklist, nor are
-----------------------------------------------------------------

[2] PDF PAGE 25
the Core is composed of four functions: GOVERN, MAP, MEASURE,
and MANAGE. Each of these high-level functions is broken down into categories a

Only Q07 remains: What does valid and reliable mean in the context of AI systems?

In [12]:
import pandas as pd
from pathlib import Path

csv_path = Path("/content/rag_from_scratch/evaluation/questions_15.csv")

df = pd.read_csv(csv_path)

page_text = next(
    p["text"] for p in page_texts
    if p["page"] == 18
)

start = page_text.index("3.1\nValid and Reliable")
end = page_text.index(
    "including the entire lifetime of\n"
    "the system.",
    start
) + len("including the entire lifetime of\n"
        "the system.")

passage = page_text[start:end]

q07 = {
    "question_id": "Q07",
    "question": "What does valid and reliable mean in the context of AI systems?",
    "gold_page": 18,
    "gold_passage": passage,
    "start_char": start,
    "end_char": end,
    "verified": True,
}

df = df[df["question_id"] != "Q07"]
df = pd.concat([df, pd.DataFrame([q07])], ignore_index=True)
df = df.sort_values("question_id").reset_index(drop=True)

df.to_csv(csv_path, index=False)

print("========== FINAL GROUND TRUTH ==========")
print(df[["question_id", "gold_page", "verified"]].to_string(index=False))
print()
print("Total questions:", len(df))
print("Verified:", df["verified"].sum())
print("Unique IDs:", df["question_id"].nunique())
print()
print("Q07 passage:")
print(passage)

assert len(df) == 15
assert df["verified"].all()
assert df["question_id"].nunique() == 15

print("\nGROUND TRUTH VALIDATION PASSED")

========== FINAL GROUND TRUTH ==========
question_id  gold_page  verified
        Q01         25      True
        Q02         27      True
        Q03         29      True
        Q04         33      True
        Q05         37      True
        Q06         17      True
        Q07         18      True
        Q08         19      True
        Q09         20      True
        Q10         21      True
        Q11         22      True
        Q12         22      True
        Q13         38      True
        Q14         20      True
        Q15         25      True

Total questions: 15
Verified: 15
Unique IDs: 15

Q07 passage:
3.1
Valid and Reliable
Validation is the “confirmation, through the provision of objective evidence, that the re-
quirements for a specific intended use or application have been fulfilled” (Source: ISO
9000:2015). Deployment of AI systems which are inaccurate, unreliable, or poorly gener-
alized to data and settings beyond their training creates and increases negati

## Retrieval evaluation

In [13]:
import re
import json
import numpy as np
import pandas as pd
from pathlib import Path
from difflib import SequenceMatcher

BASE = Path("/content/rag_from_scratch")
EVAL_DIR = BASE / "evaluation"
RESULTS_DIR = BASE / "results"

EVAL_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# --------------------------------------------------
# 1. Load verified questions
# --------------------------------------------------

questions = pd.read_csv(EVAL_DIR / "questions_15.csv")

assert len(questions) == 15
assert questions["question_id"].nunique() == 15
assert questions["verified"].astype(str).str.lower().eq("true").all()

# --------------------------------------------------
# 2. Normalize text for evidence matching
# --------------------------------------------------

def normalize_words(text):
    text = str(text).lower()

    # Repair PDF line-break hyphenation:
    # "re-\nquirements" -> "requirements"
    text = re.sub(r"-\s*\n\s*", "", text)

    # Normalize punctuation and whitespace
    text = re.sub(r"[^a-z0-9]+", " ", text)

    return text.split()


def evidence_match(gold_passage, retrieved_passage, min_words=12):
    """
    Evidence-location proxy.

    Checks whether the gold passage and retrieved
    chunk share a sufficiently long contiguous
    sequence of normalized words.

    This is stricter than page-number matching,
    but does NOT prove answer sufficiency.
    """

    gold_words = normalize_words(gold_passage)
    retrieved_words = normalize_words(retrieved_passage)

    matcher = SequenceMatcher(
        None,
        gold_words,
        retrieved_words,
        autojunk=False
    )

    longest = matcher.find_longest_match(
        0, len(gold_words),
        0, len(retrieved_words)
    ).size

    return {
        "matched": longest >= min_words,
        "longest_matching_words": longest,
        "gold_word_count": len(gold_words),
        "retrieved_word_count": len(retrieved_words),
    }


# --------------------------------------------------
# 3. Evaluate each chunking strategy
# --------------------------------------------------

all_results = []
retrieval_details = []

for chunk_set in ["small", "large"]:

    print("\n" + "=" * 65)
    print("EVALUATING:", chunk_set.upper())
    print("=" * 65)

    for _, row in questions.iterrows():

        question_id = row["question_id"]
        question = row["question"]
        gold_page = int(row["gold_page"])
        gold_passage = row["gold_passage"]

        # Use our previously implemented NumPy retriever
        retrieved = retrieve(
            question,
            chunk_set=chunk_set,
            top_k=5
        )

        evidence_hits = []
        page_hits = []

        for rank, item in enumerate(retrieved, start=1):

            page = int(item["page"])
            passage = item["text"]

            match = evidence_match(
                gold_passage,
                passage
            )

            # Require BOTH correct page and text overlap
            is_evidence_hit = (
                page == gold_page
                and match["matched"]
            )

            is_page_hit = page == gold_page

            evidence_hits.append(is_evidence_hit)
            page_hits.append(is_page_hit)

            retrieval_details.append({
                "question_id": question_id,
                "question": question,
                "chunk_set": chunk_set,
                "rank": rank,
                "chunk_id": item["chunk_id"],
                "retrieved_page": page,
                "gold_page": gold_page,
                "similarity_score": float(item["score"]),
                "page_match": is_page_hit,
                "evidence_overlap_match": is_evidence_hit,
                "longest_matching_words": (
                    match["longest_matching_words"]
                ),
                "retrieved_text": passage,
                "gold_passage": gold_passage,
            })

        first_hit_rank = next(
            (
                i + 1
                for i, hit in enumerate(evidence_hits)
                if hit
            ),
            None
        )

        reciprocal_rank = (
            1.0 / first_hit_rank
            if first_hit_rank is not None
            else 0.0
        )

        result = {
            "question_id": question_id,
            "chunk_set": chunk_set,
            "gold_page": gold_page,
            "recall_at_1": int(any(evidence_hits[:1])),
            "recall_at_3": int(any(evidence_hits[:3])),
            "recall_at_5": int(any(evidence_hits[:5])),
            "mrr": reciprocal_rank,
            "first_hit_rank": first_hit_rank,
            "page_hit_at_5": int(any(page_hits[:5])),
        }

        all_results.append(result)

        status = "HIT" if first_hit_rank else "MISS"

        print(
            f"{question_id} | "
            f"{status:4s} | "
            f"First evidence rank: {str(first_hit_rank):4s} | "
            f"Gold page: {gold_page}"
        )

# --------------------------------------------------
# 4. Build metrics table
# --------------------------------------------------

results_df = pd.DataFrame(all_results)
details_df = pd.DataFrame(retrieval_details)

summary = (
    results_df
    .groupby("chunk_set", as_index=False)
    .agg({
        "recall_at_1": "mean",
        "recall_at_3": "mean",
        "recall_at_5": "mean",
        "mrr": "mean",
        "page_hit_at_5": "mean",
    })
)

summary = summary.rename(columns={
    "recall_at_1": "Recall@1",
    "recall_at_3": "Recall@3",
    "recall_at_5": "Recall@5",
    "mrr": "MRR@5",
    "page_hit_at_5": "PageHit@5",
})

summary["chunk_size"] = summary["chunk_set"].map({
    "small": 128,
    "large": 224,
})

summary["overlap"] = summary["chunk_set"].map({
    "small": 24,
    "large": 32,
})

summary = summary[
    [
        "chunk_set",
        "chunk_size",
        "overlap",
        "Recall@1",
        "Recall@3",
        "Recall@5",
        "MRR@5",
        "PageHit@5",
    ]
]

# --------------------------------------------------
# 5. Save reproducible evaluation results
# --------------------------------------------------

results_df.to_csv(
    EVAL_DIR / "retrieval_per_question.csv",
    index=False
)

details_df.to_csv(
    EVAL_DIR / "retrieval_top5_details.csv",
    index=False
)

summary.to_csv(
    EVAL_DIR / "retrieval_metrics.csv",
    index=False
)

# --------------------------------------------------
# 6. Display final comparison
# --------------------------------------------------

print("\n" + "=" * 65)
print("RETRIEVAL PERFORMANCE COMPARISON")
print("=" * 65)

print(
    summary.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)

print("\n" + "=" * 65)
print("EVIDENCE-OVERLAP MISSES")
print("=" * 65)

misses = results_df[
    results_df["recall_at_5"] == 0
]

if misses.empty:
    print("No evidence-overlap misses in top 5.")
else:
    print(
        misses[
            ["question_id", "chunk_set", "gold_page"]
        ].to_string(index=False)
    )

print("\nSaved files:")
print(EVAL_DIR / "retrieval_per_question.csv")
print(EVAL_DIR / "retrieval_top5_details.csv")
print(EVAL_DIR / "retrieval_metrics.csv")

print("\nEVALUATION COMPLETED")


EVALUATING: SMALL
Q01 | HIT  | First evidence rank: 1    | Gold page: 25
Q02 | HIT  | First evidence rank: 3    | Gold page: 27
Q03 | HIT  | First evidence rank: 1    | Gold page: 29
Q04 | HIT  | First evidence rank: 5    | Gold page: 33
Q05 | HIT  | First evidence rank: 2    | Gold page: 37
Q06 | HIT  | First evidence rank: 1    | Gold page: 17
Q07 | HIT  | First evidence rank: 1    | Gold page: 18
Q08 | HIT  | First evidence rank: 4    | Gold page: 19
Q09 | HIT  | First evidence rank: 1    | Gold page: 20
Q10 | HIT  | First evidence rank: 1    | Gold page: 21
Q11 | HIT  | First evidence rank: 1    | Gold page: 22
Q12 | HIT  | First evidence rank: 1    | Gold page: 22
Q13 | HIT  | First evidence rank: 1    | Gold page: 38
Q14 | HIT  | First evidence rank: 1    | Gold page: 20
Q15 | MISS | First evidence rank: None | Gold page: 25

EVALUATING: LARGE
Q01 | HIT  | First evidence rank: 1    | Gold page: 25
Q02 | HIT  | First evidence rank: 2    | Gold page: 27
Q03 | HIT  | First evidence

#### Investigate retrieval failures

In [14]:
import pandas as pd
from pathlib import Path

BASE = Path("/content/rag_from_scratch")
EVAL_DIR = BASE / "evaluation"

questions = pd.read_csv(EVAL_DIR / "questions_15.csv")
details = pd.read_csv(EVAL_DIR / "retrieval_top5_details.csv")

# Q15: missed by both configurations
# Q04 and Q08: relevant evidence appeared late in top 5
inspect_ids = ["Q15", "Q04", "Q08"]

for qid in inspect_ids:

    q = questions.loc[
        questions["question_id"] == qid
    ].iloc[0]

    print("\n" + "=" * 85)
    print(f"{qid}: {q['question']}")
    print("=" * 85)

    print(f"\nGOLD PAGE: {q['gold_page']}")
    print("\nVERIFIED GOLD PASSAGE:")
    print(str(q["gold_passage"])[:1400])

    for chunk_set in ["small", "large"]:

        print("\n" + "-" * 85)
        print(f"{chunk_set.upper()} CHUNKS — TOP 5")
        print("-" * 85)

        rows = details[
            (details["question_id"] == qid) &
            (details["chunk_set"] == chunk_set)
        ].sort_values("rank")

        for _, item in rows.iterrows():

            print(
                f"\nRANK {int(item['rank'])}"
                f" | Page {int(item['retrieved_page'])}"
                f" | Score {item['similarity_score']:.4f}"
                f" | Evidence match: {item['evidence_overlap_match']}"
            )

            print(str(item["retrieved_text"])[:900])

    if qid == "Q15":

        # Look beyond top 5 to locate the gold evidence
        for chunk_set in ["small", "large"]:

            print("\n" + "-" * 85)
            print(f"Q15 — {chunk_set.upper()} — TOP 15 RANKING")
            print("-" * 85)

            extended = retrieve(
                q["question"],
                chunk_set=chunk_set,
                top_k=15
            )

            for rank, item in enumerate(extended, start=1):

                match = evidence_match(
                    q["gold_passage"],
                    item["text"]
                )

                print(
                    f"Rank {rank:2d}"
                    f" | Page {int(item['page']):2d}"
                    f" | Score {float(item['score']):.4f}"
                    f" | Gold-page: {int(item['page']) == int(q['gold_page'])}"
                    f" | Evidence-overlap: {match['matched']}"
                )

print("\nFAILURE INSPECTION COMPLETED")


Q15: Why is AI risk management considered a continuous process?

GOLD PAGE: 25

VERIFIED GOLD PASSAGE:
a checklist, nor are they necessarily an ordered set of steps.
Fig. 5. Functions organize AI risk management activities at their highest level to govern, map,
measure, and manage AI risks. Governance is designed to be a cross-cutting function to inform
and be infused throughout the other three functions.
Risk management should be continuous, timely, and performed throughout the AI system
lifecycle dimensions. AI RMF Core functions should be carried out in a way that reflects
diverse and multidisciplinary perspectives, potentially

-------------------------------------------------------------------------------------
SMALL CHUNKS — TOP 5
-------------------------------------------------------------------------------------

RANK 1 | Page 8 | Score 0.6109 | Evidence match: False
stages of organizations ’ ai risk management pro - cesses and procedures, the map, measure, and manage functio

Q15 is a genuine gold-passage ranking failure, but not necessarily a complete failure to retrieve useful information. Q04 and Q08 reveal a different problem: the retriever finds relevant material but sometimes ranks less direct passages above it.

## Audit answer-support quality

In [15]:
import pandas as pd
from pathlib import Path

eval_dir = Path("/content/rag_from_scratch/evaluation")

questions = pd.read_csv(eval_dir / "questions_15.csv")
details = pd.read_csv(eval_dir / "retrieval_top5_details.csv")

for _, q in questions.iterrows():
    qid = q["question_id"]

    print("\n" + "=" * 75)
    print(qid, "|", q["question"])
    print("=" * 75)

    for chunk_set in ["small", "large"]:
        rows = details[
            (details["question_id"] == qid) &
            (details["chunk_set"] == chunk_set)
        ].sort_values("rank")

        top1 = rows.iloc[0]

        print(f"\n{chunk_set.upper()} | TOP 1")
        print("Page:", int(top1["retrieved_page"]))
        print("Text:", str(top1["retrieved_text"])[:700])

        matches = rows[
            rows["evidence_overlap_match"]
            .astype(str).str.lower().eq("true")
        ]

        if matches.empty:
            print("Gold evidence: NOT FOUND in Top 5")
        else:
            first = matches.iloc[0]
            print(
                "Gold evidence rank:",
                int(first["rank"])
            )
            print(
                "Gold-matching text:",
                str(first["retrieved_text"])[:700]
            )

print("\nANSWER-SUPPORT AUDIT DATA READY")


Q01 | What are the four core functions of the AI RMF?

SMALL | TOP 1
Page: 25
Text: nist ai 100 - 1 ai rmf 1. 0 part 2 : core and profiles 5. ai rmf core the ai rmf core provides outcomes and actions that enable dialogue, understanding, and activities to manage ai risks and responsibly develop trustworthy ai systems. as illus - trated in figure 5, the core is composed of four functions : govern, map, measure, and manage. each of these high - level functions is broken down into categories and sub - categories. categories and subcategories are subdivided into specific actions and outcomes. actions do not constitute a checklist, nor are they necessarily an ordered
Gold evidence rank: 1
Gold-matching text: nist ai 100 - 1 ai rmf 1. 0 part 2 : core and profiles 5. ai rmf core the ai rmf core provides outcomes and actions that enable dialogue, understanding, and activities to manage ai risks and responsibly develop trustworthy ai systems. as illus - trated in figure 5, the core is composed 

observations:


Q04 — Chunk size affects answer completeness.

Q15 — A legitimate ranking failure with alternative evidence.

## Final targeted evidence check

In [16]:
import pandas as pd
from pathlib import Path

eval_dir = Path("/content/rag_from_scratch/evaluation")
details = pd.read_csv(eval_dir / "retrieval_top5_details.csv")

audit_cases = {
    "Q04": {"small": [1, 5], "large": [1, 3]},
    "Q05": {"small": [2], "large": [4]},
    "Q10": {"small": [1], "large": [1]},
    "Q12": {"small": [1], "large": [1]},
    "Q15": {"small": [1, 2], "large": [2, 3]},
}

for qid, configurations in audit_cases.items():

    question = details[
        details["question_id"] == qid
    ]["question"].iloc[0]

    print("\n" + "=" * 90)
    print(qid, "-", question)
    print("=" * 90)

    for chunk_set, ranks in configurations.items():

        for rank in ranks:

            matches = details[
                (details["question_id"] == qid)
                & (details["chunk_set"] == chunk_set)
                & (details["rank"] == rank)
            ]

            if matches.empty:
                continue

            item = matches.iloc[0]

            print(
                f"\n[{chunk_set.upper()}] "
                f"RANK {rank} | "
                f"PAGE {int(item['retrieved_page'])}"
            )

            print("-" * 75)
            print(item["retrieved_text"])

print("\nTARGETED EVIDENCE AUDIT COMPLETE")


Q04 - What is the purpose of the MEASURE function?

[SMALL] RANK 1 | PAGE 33
---------------------------------------------------------------------------
, may need to be developed. the degree to which each measurement type provides unique and meaningful information to the assessment of ai risks should be considered. framework users will en - hance their capacity to comprehensively evaluate system trustworthiness, identify and track existing and emergent risks, and verify efficacy of the metrics. measurement outcomes will be utilized in the manage function to assist risk monitoring and response efforts. it is in - cumbent on framework users to continue applying the measure function to ai systems as knowledge, methodologies, risks, and impacts evolve over time. page 28

[SMALL] RANK 5 | PAGE 33
---------------------------------------------------------------------------
nist ai 100 - 1 ai rmf 1. 0 table 2 : categories and subcategories for the map function. ( continued ) map 5. 2 : pract

Failure 1 — Q15: Semantic ranking failure

Failure 2 — Q10: Chunk-boundary and answer-completeness failure



#### Save the failure analysis

In [17]:
import json
from pathlib import Path

output_path = Path(
    "/content/rag_from_scratch/evaluation/failure_analysis.json"
)

failure_analysis = {
    "evaluation_method": (
        "Dense cosine retrieval with manually verified "
        "source passages and an automated evidence-overlap "
        "proxy. Failure cases manually inspected."
    ),
    "failures": [
        {
            "question_id": "Q15",
            "question": (
                "Why is AI risk management considered "
                "a continuous process?"
            ),
            "failure_type": "Gold-passage ranking failure",
            "gold_page": 25,
            "small_gold_rank": 14,
            "large_gold_rank": 12,
            "observation": (
                "Both chunk sizes rank the annotated passage "
                "outside the top five. The large configuration "
                "does retrieve alternative supporting evidence "
                "on pages 36 and 15."
            ),
            "proposed_fixes": [
                "Combine dense retrieval with BM25.",
                "Re-rank candidates using query-passage relevance.",
                "Evaluate alternative valid supporting passages."
            ]
        },
        {
            "question_id": "Q10",
            "question": (
                "What is the difference between "
                "explainability and interpretability?"
            ),
            "failure_type": (
                "Top-ranked chunk lacks complete answer"
            ),
            "gold_page": 21,
            "small_top1_answer_sufficient": True,
            "large_top1_answer_sufficient": False,
            "observation": (
                "The 128-token top-ranked chunk explicitly "
                "defines both concepts. The 224-token top-ranked "
                "chunk discusses their benefits but omits "
                "the explicit distinction."
            ),
            "proposed_fixes": [
                "Use section-aware chunking.",
                "Preserve definitions within chunk boundaries.",
                "Use answer-aware re-ranking."
            ]
        }
    ],
    "limitations": [
        "Only 15 evaluation questions.",
        "Single 48-page source document.",
        "Evidence-overlap matching is not equivalent "
        "to answer sufficiency.",
        "Proposed fixes have not yet been experimentally tested."
    ]
}

output_path.parent.mkdir(
    parents=True,
    exist_ok=True
)

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(
        failure_analysis,
        f,
        indent=2,
        ensure_ascii=False
    )

print("Saved:", output_path)

for failure in failure_analysis["failures"]:
    print(
        failure["question_id"],
        "-",
        failure["failure_type"]
    )

print("\nFAILURE ANALYSIS SAVED")

Saved: /content/rag_from_scratch/evaluation/failure_analysis.json
Q15 - Gold-passage ranking failure
Q10 - Top-ranked chunk lacks complete answer

FAILURE ANALYSIS SAVED


## BM25 and re-ranking

**What would BM25 add?**

Our current retriever uses MiniLM embeddings and cosine similarity. It searches for passages that are semantically similar to the question.

BM25 is a lexical retrieval method. It ranks passages using matching words, term frequency, document frequency, and document-length normalization.

For example, in Q15, the embedding retriever ranked the exact gold passage outside the top five. BM25 could help prioritize passages containing terms such as risk management, continuous, and lifecycle.

Why combine them? Dense retrieval recognizes semantic similarity, while BM25 rewards explicit terminology. Hybrid retrieval can combine both candidate lists using Reciprocal Rank Fusion (RRF).

This is a proposed improvement, not a measured performance gain.

**What would re-ranking add?**

A re-ranker evaluates a question and candidate passage together, rather than comparing independently computed embeddings.

Consider Q10:

Question: What is the difference between explainability and interpretability?

The 224-token retriever ranked a passage first that discussed the benefits of both concepts but did not explicitly distinguish their definitions.

A suitable cross-encoder re-ranker could favor the passage that directly defines both concepts.

Trade-off: Re-ranking adds inference latency because it scores candidate pairs. However, it is practical when applied to a relatively small shortlist, such as the top 10–20 passages.

## End-to-end RAG demonstration

In [18]:
import re

def rag_answer(question, chunk_set="small", top_k=3):
    """
    Lightweight retrieval-grounded QA demonstration.

    Retrieves evidence and extracts relevant sentences.
    No external LLM or API key required.
    """

    results = retrieve(
        question,
        chunk_set=chunk_set,
        top_k=top_k
    )

    question_terms = set(
        re.findall(r"\b[a-z]{3,}\b", question.lower())
    )

    stopwords = {
        "what", "which", "when", "where", "why",
        "how", "does", "the", "are", "for",
        "and", "with", "from", "into", "about"
    }

    question_terms -= stopwords

    candidates = []

    for item in results:

        text = item["text"]

        sentences = re.split(
            r"(?<=[.!?])\s+",
            text
        )

        for sentence in sentences:

            sentence = sentence.strip()

            if len(sentence) < 35:
                continue

            words = set(
                re.findall(
                    r"\b[a-z]{3,}\b",
                    sentence.lower()
                )
            )

            lexical_overlap = len(
                question_terms & words
            )

            candidates.append({
                "sentence": sentence,
                "page": item["page"],
                "chunk_id": item["chunk_id"],
                "score": (
                    lexical_overlap
                    + 0.1 * float(item["score"])
                )
            })

    candidates.sort(
        key=lambda x: x["score"],
        reverse=True
    )

    selected = []
    seen = set()

    for candidate in candidates:

        sentence = candidate["sentence"]

        if sentence in seen:
            continue

        selected.append(candidate)
        seen.add(sentence)

        if len(selected) == 2:
            break

    print("=" * 75)
    print("QUESTION")
    print(question)

    print("\nEXTRACTED EVIDENCE")
    print(
        "This is extractive evidence, "
        "not an LLM-generated answer."
    )

    if not selected:
        print("Insufficient evidence retrieved.")
    else:
        for candidate in selected:
            print(
                f"\n{candidate['sentence']}"
                f" [PDF page {candidate['page']}]"
            )

    print("\nTOP RETRIEVED SOURCES")

    for item in results:
        print(
            f"Rank {item['rank']} | "
            f"PDF page {item['page']} | "
            f"Chunk {item['chunk_id']} | "
            f"Similarity {item['score']:.4f}"
        )

    return {
        "question": question,
        "evidence": selected,
        "retrieved_sources": results
    }


# --------------------------------------------------
# Demonstration
# --------------------------------------------------

demo = rag_answer(
    "What are the four core functions of the AI RMF?",
    chunk_set="small",
    top_k=3
)

QUESTION
What are the four core functions of the AI RMF?

EXTRACTED EVIDENCE
This is extractive evidence, not an LLM-generated answer.

as illus - trated in figure 5, the core is composed of four functions : govern, map, measure, and manage. [PDF page 25]

ai rmf core functions should be carried out in a way that reflects diverse and multidisciplinary perspectives, potentially including the views of ai actors out - side the organization [PDF page 25]

TOP RETRIEVED SOURCES
Rank 1 | PDF page 25 | Chunk p25_t0 | Similarity 0.7781
Rank 2 | PDF page 47 | Chunk p47_t0 | Similarity 0.6882
Rank 3 | PDF page 25 | Chunk p25_t104 | Similarity 0.6857


## Run the end-to-end system on all 15 questions

In [19]:
import json
import pandas as pd
from pathlib import Path

BASE = Path("/content/rag_from_scratch")
EVAL_DIR = BASE / "evaluation"

questions_df = pd.read_csv(
    EVAL_DIR / "questions_15.csv"
)

assert len(questions_df) == 15

answer_records = []

for _, row in questions_df.iterrows():

    qid = row["question_id"]
    question = row["question"]

    print("\n" + "=" * 85)
    print(f"{qid}: {question}")
    print("=" * 85)

    result = rag_answer(
        question,
        chunk_set="small",
        top_k=5
    )

    extracted = result["evidence"]

    answer_text = " ".join(
        item["sentence"]
        for item in extracted
    )

    source_pages = sorted(set(
        int(item["page"])
        for item in extracted
    ))

    answer_records.append({
        "question_id": qid,
        "question": question,
        "gold_page": int(row["gold_page"]),
        "answer": answer_text,
        "source_pages": source_pages,
        "top_retrieved_page": (
            int(result["retrieved_sources"][0]["page"])
            if result["retrieved_sources"]
            else None
        ),
        "answer_review_status": "pending"
    })

output_json = EVAL_DIR / "rag_answers_15.json"
output_csv = EVAL_DIR / "rag_answers_15.csv"

with open(output_json, "w", encoding="utf-8") as f:
    json.dump(
        answer_records,
        f,
        indent=2,
        ensure_ascii=False
    )

pd.DataFrame([
    {
        **record,
        "source_pages": ", ".join(
            map(str, record["source_pages"])
        )
    }
    for record in answer_records
]).to_csv(
    output_csv,
    index=False
)

print("\n" + "=" * 85)
print("END-TO-END TEST SUMMARY")
print("=" * 85)

print("Questions processed:", len(answer_records))
print("Nonempty answers:", sum(
    bool(record["answer"].strip())
    for record in answer_records
))
print("Answers manually verified: 0")
print("Review status: PENDING")

print("\nSaved:")
print(output_json)
print(output_csv)

print("\nEND-TO-END TEST COMPLETED")


Q01: What are the four core functions of the AI RMF?
QUESTION
What are the four core functions of the AI RMF?

EXTRACTED EVIDENCE
This is extractive evidence, not an LLM-generated answer.

as illus - trated in figure 5, the core is composed of four functions : govern, map, measure, and manage. [PDF page 25]

ai rmf core functions should be carried out in a way that reflects diverse and multidisciplinary perspectives, potentially including the views of ai actors out - side the organization [PDF page 25]

TOP RETRIEVED SOURCES
Rank 1 | PDF page 25 | Chunk p25_t0 | Similarity 0.7781
Rank 2 | PDF page 47 | Chunk p47_t0 | Similarity 0.6882
Rank 3 | PDF page 25 | Chunk p25_t104 | Similarity 0.6857
Rank 4 | PDF page 37 | Chunk p37_t0 | Similarity 0.6250
Rank 5 | PDF page 44 | Chunk p44_t208 | Similarity 0.6155

Q02: What is the purpose of the GOVERN function?
QUESTION
What is the purpose of the GOVERN function?

EXTRACTED EVIDENCE
This is extractive evidence, not an LLM-generated answer.

af

## Improve evidence selection

In [20]:
import re
import numpy as np

def rag_answer_v2(
    question,
    chunk_set="small",
    top_k=5,
    max_sentences=3
):
    """
    Retrieval-grounded extractive QA.

    1. Retrieve top-k chunks.
    2. Split chunks into candidate sentences.
    3. Embed question and sentences.
    4. Rank sentences by semantic similarity.
    5. Return evidence with PDF citations.
    """

    retrieved = retrieve(
        question,
        chunk_set=chunk_set,
        top_k=top_k
    )

    candidates = []
    seen = set()

    for item in retrieved:

        sentences = re.split(
            r"(?<=[.!?])\s+",
            item["text"]
        )

        for sentence in sentences:

            sentence = sentence.strip()

            if len(sentence) < 45:
                continue

            normalized = " ".join(
                sentence.lower().split()
            )

            if normalized in seen:
                continue

            seen.add(normalized)

            candidates.append({
                "sentence": sentence,
                "page": int(item["page"]),
                "chunk_id": item["chunk_id"],
                "retrieval_rank": int(item["rank"]),
                "retrieval_score": float(item["score"])
            })

    if not candidates:
        return {
            "question": question,
            "answer": "",
            "evidence": [],
            "retrieved_sources": retrieved
        }

    sentence_texts = [
        item["sentence"]
        for item in candidates
    ]

    question_embedding = model.encode(
        [question],
        normalize_embeddings=True
    )[0]

    sentence_embeddings = model.encode(
        sentence_texts,
        normalize_embeddings=True,
        batch_size=32
    )

    similarities = (
        sentence_embeddings @ question_embedding
    )

    for i, item in enumerate(candidates):

        semantic_score = float(similarities[i])

        # Semantic relevance dominates.
        # Original retrieval score is a small tie-breaker.
        item["semantic_score"] = semantic_score
        item["combined_score"] = (
            0.90 * semantic_score
            + 0.10 * item["retrieval_score"]
        )

    candidates.sort(
        key=lambda x: x["combined_score"],
        reverse=True
    )

    selected = candidates[:max_sentences]

    answer = " ".join(
        item["sentence"]
        for item in selected
    )

    return {
        "question": question,
        "answer": answer,
        "evidence": selected,
        "retrieved_sources": retrieved
    }


# Test the questions that previously had weak answers.

test_questions = {
    "Q02": "What is the purpose of the GOVERN function?",
    "Q04": "What is the purpose of the MEASURE function?",
    "Q07": "What does valid and reliable mean in the context of AI systems?",
    "Q11": "How does the framework address privacy in AI systems?",
    "Q15": "Why is AI risk management considered a continuous process?"
}

for qid, question in test_questions.items():

    result = rag_answer_v2(
        question,
        chunk_set="small",
        top_k=5,
        max_sentences=3
    )

    print("\n" + "=" * 85)
    print(qid, "-", question)
    print("=" * 85)

    for item in result["evidence"]:
        print(
            f"\n[PDF page {item['page']}] "
            f"Semantic score: {item['semantic_score']:.4f}"
        )
        print(item["sentence"])

print("\nSEMANTIC ANSWER SELECTION TEST COMPLETE")


Q02 - What is the purpose of the GOVERN function?

[PDF page 5] Semantic score: 0.5662
governance is designed to be a cross - cutting function to inform and be infused throughout the other three functions.

[PDF page 26] Semantic score: 0.4955
governance struc - ture is in place, functions may be performed in any order across the ai lifecycle as deemed to add value by a user of the framework.

[PDF page 26] Semantic score: 0.4631
after instituting the outcomes in govern, most users of the ai rmf would start with the map function and con - tinue to measure or manage.

Q04 - What is the purpose of the MEASURE function?

[PDF page 35] Semantic score: 0.6602
measure function are evaluated and documented.

[PDF page 30] Semantic score: 0.5392
outcomes in the map function are the basis for the measure and manage functions.

[PDF page 33] Semantic score: 0.4975
3 measure the measure function employs quantitative, qualitative, or mixed - method tools, tech - niques, and methodologies to analy

## Implement BM25 from scratch and hybrid retrieval

In [21]:
import re
import math
import numpy as np
from collections import Counter


# ============================================================
# 1. BM25 IMPLEMENTATION FROM SCRATCH
# ============================================================

def bm25_tokenize(text):
    """
    Simple lexical tokenizer.
    """
    return re.findall(
        r"[a-z0-9]+",
        text.lower()
    )


class BM25FromScratch:

    def __init__(
        self,
        documents,
        k1=1.5,
        b=0.75
    ):

        self.documents = documents
        self.k1 = k1
        self.b = b

        self.tokenized = [
            bm25_tokenize(doc)
            for doc in documents
        ]

        self.N = len(self.tokenized)

        self.doc_lengths = np.array(
            [len(doc) for doc in self.tokenized],
            dtype=np.float32
        )

        self.avgdl = float(
            np.mean(self.doc_lengths)
        )

        self.term_frequencies = [
            Counter(doc)
            for doc in self.tokenized
        ]

        document_frequencies = Counter()

        for tokens in self.tokenized:
            document_frequencies.update(
                set(tokens)
            )

        self.idf = {}

        for term, df in document_frequencies.items():

            self.idf[term] = math.log(
                1 + (
                    self.N - df + 0.5
                ) / (
                    df + 0.5
                )
            )

    def score(self, query):

        query_terms = bm25_tokenize(query)

        scores = np.zeros(
            self.N,
            dtype=np.float32
        )

        for term in query_terms:

            if term not in self.idf:
                continue

            idf = self.idf[term]

            for i, frequencies in enumerate(
                self.term_frequencies
            ):

                tf = frequencies.get(term, 0)

                if tf == 0:
                    continue

                denominator = (
                    tf
                    + self.k1 * (
                        1 - self.b
                        + self.b
                        * self.doc_lengths[i]
                        / max(self.avgdl, 1e-9)
                    )
                )

                scores[i] += (
                    idf
                    * tf
                    * (self.k1 + 1)
                    / denominator
                )

        return scores


# ============================================================
# 2. BUILD BM25 INDEXES
# ============================================================

bm25_indexes = {}

for chunk_set in ["small", "large"]:

    documents = [
        chunk["text"]
        for chunk in all_chunks[chunk_set]
    ]

    bm25_indexes[chunk_set] = (
        BM25FromScratch(documents)
    )

    print(
        f"{chunk_set.upper()} BM25 index:",
        len(documents),
        "chunks"
    )


# ============================================================
# 3. HYBRID RETRIEVAL WITH RRF
# ============================================================

def hybrid_retrieve(
    query,
    chunk_set="small",
    top_k=5,
    candidate_k=30,
    rrf_k=60
):
    """
    Combine dense cosine similarity and BM25 rankings
    using Reciprocal Rank Fusion.
    """

    chunks = all_chunks[chunk_set]

    embeddings = (
        small_embeddings
        if chunk_set == "small"
        else large_embeddings
    )

    # Dense scores
    query_embedding = model.encode(
        [query],
        normalize_embeddings=True
    )[0]

    dense_scores = (
        embeddings @ query_embedding
    )

    # BM25 scores
    lexical_scores = (
        bm25_indexes[chunk_set].score(query)
    )

    n = len(chunks)

    candidate_k = min(
        candidate_k,
        n
    )

    dense_order = np.argsort(
        -dense_scores
    )[:candidate_k]

    bm25_order = np.argsort(
        -lexical_scores
    )[:candidate_k]

    # Reciprocal Rank Fusion
    fused_scores = {}

    for rank, idx in enumerate(
        dense_order,
        start=1
    ):

        idx = int(idx)

        fused_scores[idx] = (
            fused_scores.get(idx, 0.0)
            + 1.0 / (rrf_k + rank)
        )

    for rank, idx in enumerate(
        bm25_order,
        start=1
    ):

        idx = int(idx)

        fused_scores[idx] = (
            fused_scores.get(idx, 0.0)
            + 1.0 / (rrf_k + rank)
        )

    final_order = sorted(
        fused_scores,
        key=lambda idx: (
            -fused_scores[idx],
            idx
        )
    )[:top_k]

    results = []

    for rank, idx in enumerate(
        final_order,
        start=1
    ):

        chunk = chunks[idx]

        results.append({
            "rank": rank,
            "chunk_id": chunk["chunk_id"],
            "page": chunk["page"],
            "score": float(
                fused_scores[idx]
            ),
            "dense_score": float(
                dense_scores[idx]
            ),
            "bm25_score": float(
                lexical_scores[idx]
            ),
            "text": chunk["text"]
        })

    return results


# ============================================================
# 4. TEST HYBRID RETRIEVAL
# ============================================================

test_questions = {
    "Q04": "What is the purpose of the MEASURE function?",
    "Q10": "What is the difference between explainability and interpretability?",
    "Q15": "Why is AI risk management considered a continuous process?"
}

for qid, question in test_questions.items():

    print("\n" + "=" * 85)
    print(qid, "-", question)
    print("=" * 85)

    for chunk_set in ["small", "large"]:

        results = hybrid_retrieve(
            question,
            chunk_set=chunk_set,
            top_k=5
        )

        print(
            f"\n{chunk_set.upper()} HYBRID TOP 5"
        )

        for item in results:

            print(
                f"Rank {item['rank']} | "
                f"Page {item['page']} | "
                f"RRF {item['score']:.5f} | "
                f"Dense {item['dense_score']:.4f} | "
                f"BM25 {item['bm25_score']:.4f}"
            )

            print(
                item["text"][:220],
                "\n"
            )

print("\nHYBRID RETRIEVAL TEST COMPLETE")

SMALL BM25 index: 212 chunks
LARGE BM25 index: 126 chunks

Q04 - What is the purpose of the MEASURE function?

SMALL HYBRID TOP 5
Rank 1 | Page 35 | RRF 0.03078 | Dense 0.2616 | BM25 6.3054
nist ai 100 - 1 ai rmf 1. 0 table 3 : categories and subcategories for the measure function. ( continued ) measure 2. 6 : the ai system is evaluated regularly for safety risks – as identified in the map function. the ai  

Rank 2 | Page 34 | RRF 0.03008 | Dense 0.2584 | BM25 5.9446
nist ai 100 - 1 ai rmf 1. 0 practices related to measuring ai risks are described in the nist ai rmf playbook. table 3 lists the measure function ’ s categories and subcategories. table 3 : categories and subcategories f 

Rank 3 | Page 33 | RRF 0.02973 | Dense 0.3662 | BM25 5.0420
, may need to be developed. the degree to which each measurement type provides unique and meaningful information to the assessment of ai risks should be considered. framework users will en - hance their capacity to compr 

Rank 4 | Page 5 | RRF

## Evaluate hybrid retrieval on all 15 questions

In [22]:
import pandas as pd
import numpy as np
from pathlib import Path

BASE = Path("/content/rag_from_scratch")
EVAL_DIR = BASE / "evaluation"

questions_df = pd.read_csv(
    EVAL_DIR / "questions_15.csv"
)

baseline_metrics = pd.read_csv(
    EVAL_DIR / "retrieval_metrics.csv"
)

hybrid_records = []
hybrid_details = []

for chunk_set in ["small", "large"]:

    for _, question_row in questions_df.iterrows():

        qid = question_row["question_id"]
        question = question_row["question"]
        gold_page = int(question_row["gold_page"])
        gold_passage = str(question_row["gold_passage"])

        retrieved = hybrid_retrieve(
            query=question,
            chunk_set=chunk_set,
            top_k=5,
            candidate_k=30,
            rrf_k=60
        )

        first_match_rank = None
        page_hit = False

        for item in retrieved:

            rank = int(item["rank"])
            page = int(item["page"])

            same_page = page == gold_page

            passage_match = (
                same_page
                and evidence_match(
                    gold_passage,
                    item["text"],
                    min_words=12
                )
            )

            if same_page:
                page_hit = True

            if passage_match and first_match_rank is None:
                first_match_rank = rank

            hybrid_details.append({
                "question_id": qid,
                "chunk_set": chunk_set,
                "rank": rank,
                "retrieved_page": page,
                "gold_page": gold_page,
                "evidence_overlap_match": bool(
                    passage_match
                ),
                "retrieved_text": item["text"]
            })

        hybrid_records.append({
            "question_id": qid,
            "chunk_set": chunk_set,
            "first_match_rank": first_match_rank,
            "Recall@1": int(
                first_match_rank is not None
                and first_match_rank <= 1
            ),
            "Recall@3": int(
                first_match_rank is not None
                and first_match_rank <= 3
            ),
            "Recall@5": int(
                first_match_rank is not None
                and first_match_rank <= 5
            ),
            "reciprocal_rank": (
                1.0 / first_match_rank
                if first_match_rank is not None
                else 0.0
            ),
            "PageHit@5": int(page_hit)
        })


hybrid_df = pd.DataFrame(hybrid_records)
hybrid_details_df = pd.DataFrame(hybrid_details)

hybrid_summary = (
    hybrid_df
    .groupby("chunk_set", as_index=False)
    .agg({
        "Recall@1": "mean",
        "Recall@3": "mean",
        "Recall@5": "mean",
        "reciprocal_rank": "mean",
        "PageHit@5": "mean"
    })
    .rename(columns={
        "reciprocal_rank": "MRR@5"
    })
)

hybrid_summary["method"] = "hybrid_rrf"

# Preserve original baseline.
baseline_comparison = baseline_metrics.copy()
baseline_comparison["method"] = "dense"

comparison = pd.concat(
    [
        baseline_comparison,
        hybrid_summary
    ],
    ignore_index=True
)

comparison = comparison[
    [
        "method",
        "chunk_set",
        "Recall@1",
        "Recall@3",
        "Recall@5",
        "MRR@5",
        "PageHit@5"
    ]
]

comparison = comparison.sort_values(
    ["chunk_set", "method"]
)

# Save all experiment outputs.
hybrid_df.to_csv(
    EVAL_DIR / "hybrid_per_question.csv",
    index=False
)

hybrid_details_df.to_csv(
    EVAL_DIR / "hybrid_top5_details.csv",
    index=False
)

comparison.to_csv(
    EVAL_DIR / "dense_vs_hybrid_comparison.csv",
    index=False
)

print("=" * 85)
print("DENSE VS HYBRID RETRIEVAL")
print("=" * 85)

print(
    comparison.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)

print("\n" + "=" * 85)
print("HYBRID PER-QUESTION RESULTS")
print("=" * 85)

print(
    hybrid_df[
        [
            "question_id",
            "chunk_set",
            "first_match_rank",
            "Recall@5"
        ]
    ].to_string(index=False)
)

print("\n" + "=" * 85)
print("Q15 FAILURE-FIX VERIFICATION")
print("=" * 85)

print(
    hybrid_df[
        hybrid_df["question_id"] == "Q15"
    ][
        [
            "chunk_set",
            "first_match_rank",
            "Recall@5"
        ]
    ].to_string(index=False)
)

print("\nSaved:")
print(EVAL_DIR / "hybrid_per_question.csv")
print(EVAL_DIR / "hybrid_top5_details.csv")
print(EVAL_DIR / "dense_vs_hybrid_comparison.csv")

print("\nFULL HYBRID EVALUATION COMPLETE")

DENSE VS HYBRID RETRIEVAL
    method chunk_set  Recall@1  Recall@3  Recall@5  MRR@5  PageHit@5
     dense     large    0.6000    0.8000    0.9333 0.7078     0.9333
hybrid_rrf     large    0.6000    0.9333    0.9333 0.7444     0.9333
     dense     small    0.6667    0.8000    0.9333 0.7522     0.9333
hybrid_rrf     small    0.7333    0.9333    0.9333 0.8000     0.9333

HYBRID PER-QUESTION RESULTS
question_id chunk_set  first_match_rank  Recall@5
        Q01     small               1.0         1
        Q02     small               1.0         1
        Q03     small               3.0         1
        Q04     small               3.0         1
        Q05     small               1.0         1
        Q06     small               1.0         1
        Q07     small               1.0         1
        Q08     small               NaN         0
        Q09     small               1.0         1
        Q10     small               1.0         1
        Q11     small               1.0         1


Save the final experiment report

In [24]:
import json
import pandas as pd
from pathlib import Path

BASE = Path("/content/rag_from_scratch")
EVAL_DIR = BASE / "evaluation"
RESULTS_DIR = BASE / "results"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================
# 1. LOAD EXISTING RESULTS
# ============================================================

questions = pd.read_csv(
    EVAL_DIR / "questions_15.csv"
)

comparison = pd.read_csv(
    EVAL_DIR / "dense_vs_hybrid_comparison.csv"
)

hybrid = pd.read_csv(
    EVAL_DIR / "hybrid_per_question.csv"
)

assert len(questions) == 15
assert len(hybrid) == 30

# ============================================================
# 2. RECONSTRUCT DENSE PER-QUESTION RESULTS
# ============================================================

dense_records = []

for chunk_set in ["small", "large"]:

    for _, row in questions.iterrows():

        qid = row["question_id"]
        question = row["question"]
        gold_page = int(row["gold_page"])
        gold_passage = str(row["gold_passage"])

        results = retrieve(
            question,
            chunk_set=chunk_set,
            top_k=5
        )

        first_match_rank = None

        for item in results:

            if (
                int(item["page"]) == gold_page
                and evidence_match(
                    gold_passage,
                    item["text"],
                    min_words=12
                )
            ):
                first_match_rank = int(item["rank"])
                break

        dense_records.append({
            "question_id": qid,
            "chunk_set": chunk_set,
            "first_match_rank": first_match_rank,
            "Recall@5": int(
                first_match_rank is not None
            )
        })

dense = pd.DataFrame(dense_records)

# ============================================================
# 3. COMPARE DENSE AND HYBRID
# ============================================================

changes = dense[
    ["question_id", "chunk_set", "Recall@5"]
].merge(
    hybrid[
        ["question_id", "chunk_set", "Recall@5"]
    ],
    on=["question_id", "chunk_set"],
    suffixes=("_dense", "_hybrid"),
    validate="one_to_one"
)

assert len(changes) == 30

improvements = changes[
    (changes["Recall@5_dense"] == 0)
    & (changes["Recall@5_hybrid"] == 1)
]

regressions = changes[
    (changes["Recall@5_dense"] == 1)
    & (changes["Recall@5_hybrid"] == 0)
]

# ============================================================
# 4. VERIFY AGAINST SAVED SUMMARY METRICS
# ============================================================

for chunk_set in ["small", "large"]:

    dense_recall = dense.loc[
        dense["chunk_set"] == chunk_set,
        "Recall@5"
    ].mean()

    saved_dense = comparison.loc[
        (comparison["method"] == "dense")
        & (comparison["chunk_set"] == chunk_set),
        "Recall@5"
    ].iloc[0]

    assert abs(dense_recall - saved_dense) < 1e-6, (
        f"Dense metric mismatch for {chunk_set}"
    )

print("Dense results verified against saved metrics.")

# ============================================================
# 5. BUILD FINAL REPORT
# ============================================================

report = {
    "project": "RAG From Scratch — NIST AI RMF",

    "document": {
        "name": "NIST AI Risk Management Framework 1.0",
        "pages": 48
    },

    "evaluation": {
        "questions": 15,
        "gold_evidence": "Manually annotated passages",
        "matching": (
            "Same PDF page and at least 12 "
            "consecutive matching normalized words"
        )
    },

    "retrieval": {
        "dense": "MiniLM embeddings with cosine similarity",
        "lexical": "BM25 implemented from scratch",
        "hybrid": "Reciprocal Rank Fusion",
        "rrf_k": 60,
        "candidate_k": 30
    },

    "comparison": comparison.to_dict(
        orient="records"
    ),

    "recovered_at_5": improvements[
        ["question_id", "chunk_set"]
    ].to_dict(orient="records"),

    "regressed_at_5": regressions[
        ["question_id", "chunk_set"]
    ].to_dict(orient="records"),

    "limitations": [
        "Only 15 evaluation questions",
        "Single 48-page source document",
        "Gold-passage overlap is not answer accuracy",
        "No statistical significance test",
        "Hybrid parameters were not independently tuned",
        "Extractive answer selection remains imperfect"
    ]
}

# ============================================================
# 6. SAVE REPORT
# ============================================================

report_path = (
    RESULTS_DIR / "final_experiment_report.json"
)

with open(
    report_path,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        report,
        f,
        indent=2,
        ensure_ascii=False
    )

changes.to_csv(
    EVAL_DIR / "dense_hybrid_question_changes.csv",
    index=False
)

# ============================================================
# 7. PRINT FINAL RESULTS
# ============================================================

print("\n" + "=" * 75)
print("FINAL EXPERIMENT SUMMARY")
print("=" * 75)

print("\nRecovered gold evidence:")
print(
    improvements[
        ["question_id", "chunk_set"]
    ].to_string(index=False)
)

print("\nNew misses:")
print(
    regressions[
        ["question_id", "chunk_set"]
    ].to_string(index=False)
)

print("\nFinal comparison:")
print(
    comparison.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)

print("\nSaved:")
print(report_path)
print(EVAL_DIR / "dense_hybrid_question_changes.csv")

print("\nFINAL EXPERIMENT REPORT COMPLETE")

Dense results verified against saved metrics.

FINAL EXPERIMENT SUMMARY

Recovered gold evidence:
question_id chunk_set
        Q15     small
        Q15     large

New misses:
question_id chunk_set
        Q08     small
        Q03     large

Final comparison:
    method chunk_set  Recall@1  Recall@3  Recall@5  MRR@5  PageHit@5
     dense     large    0.6000    0.8000    0.9333 0.7078     0.9333
hybrid_rrf     large    0.6000    0.9333    0.9333 0.7444     0.9333
     dense     small    0.6667    0.8000    0.9333 0.7522     0.9333
hybrid_rrf     small    0.7333    0.9333    0.9333 0.8000     0.9333

Saved:
/content/rag_from_scratch/results/final_experiment_report.json
/content/rag_from_scratch/evaluation/dense_hybrid_question_changes.csv

FINAL EXPERIMENT REPORT COMPLETE


## Final project audit

In [26]:
from pathlib import Path
import json
import pandas as pd

BASE = Path("/content/rag_from_scratch")

required_files = {
    "Source PDF": BASE / "data/nist_ai_rmf.pdf",
    "Small chunks": BASE / "results/chunks_small.json",
    "Large chunks": BASE / "results/chunks_large.json",
    "15 annotated questions": BASE / "evaluation/questions_15.csv",
    "Dense retrieval metrics": BASE / "evaluation/retrieval_metrics.csv",
    "Dense per-question results": BASE / "evaluation/retrieval_per_question.csv",
    "Hybrid per-question results": BASE / "evaluation/hybrid_per_question.csv",
    "Dense vs hybrid comparison": BASE / "evaluation/dense_vs_hybrid_comparison.csv",
    "Failure analysis": BASE / "evaluation/failure_analysis.json",
    "End-to-end answers": BASE / "evaluation/rag_answers_15.csv",
    "Final experiment report": BASE / "results/final_experiment_report.json",
    "Per-question changes": BASE / "evaluation/dense_hybrid_question_changes.csv",
}

print("=" * 75)
print("FINAL PROJECT ARTIFACT AUDIT")
print("=" * 75)

missing = []

for label, path in required_files.items():

    if path.exists():
        print(
            f"[OK]      {label:<32} "
            f"{path.stat().st_size:,} bytes"
        )
    else:
        print(f"[MISSING] {label}")
        missing.append(label)

print("\n" + "=" * 75)
print("DATA VALIDATION")
print("=" * 75)

questions_path = (
    BASE / "evaluation/questions_15.csv"
)

if questions_path.exists():
    questions = pd.read_csv(questions_path)

    print("Questions:", len(questions))
    print(
        "Unique question IDs:",
        questions["question_id"].nunique()
    )

    if "verified" in questions.columns:
        print(
            "Verified annotations:",
            questions["verified"].astype(str)
            .str.lower()
            .eq("true")
            .sum()
        )

report_path = (
    BASE / "results/final_experiment_report.json"
)

if report_path.exists():
    with open(
        report_path,
        encoding="utf-8"
    ) as f:
        report = json.load(f)

    print(
        "Comparison configurations:",
        len(report["comparison"])
    )

    print(
        "Recovered cases:",
        len(report["recovered_at_5"])
    )

    print(
        "Regression cases:",
        len(report["regressed_at_5"])
    )

print("\n" + "=" * 75)
print("AUDIT RESULT")
print("=" * 75)

if missing:
    print(
        f"INCOMPLETE: {len(missing)} "
        "required file(s) missing."
    )
else:
    print("PASS: All required project artifacts exist.")

print("\nNOTE: This checks saved artifacts,")
print("not clean-runtime reproducibility.")

FINAL PROJECT ARTIFACT AUDIT
[OK]      Source PDF                       2,053,443 bytes
[OK]      Small chunks                     158,158 bytes
[OK]      Large chunks                     138,485 bytes
[OK]      15 annotated questions           9,628 bytes
[OK]      Dense retrieval metrics          243 bytes
[OK]      Dense per-question results       1,012 bytes
[OK]      Hybrid per-question results      941 bytes
[OK]      Dense vs hybrid comparison       423 bytes
[OK]      Failure analysis                 1,712 bytes
[OK]      End-to-end answers               6,219 bytes
[OK]      Final experiment report          2,117 bytes
[OK]      Per-question changes             473 bytes

DATA VALIDATION
Questions: 15
Unique question IDs: 15
Verified annotations: 15
Comparison configurations: 4
Recovered cases: 2
Regression cases: 2

AUDIT RESULT
PASS: All required project artifacts exist.

NOTE: This checks saved artifacts,
not clean-runtime reproducibility.


In [27]:
from pathlib import Path
import shutil
from google.colab import files

# Project directory
project_dir = Path("/content/rag_from_scratch")

# ZIP destination
zip_path = Path("/content/RAG_From_Scratch_Final.zip")

# Verify project exists
assert project_dir.exists(), "Project folder not found!"

# Create ZIP containing the entire project
shutil.make_archive(
    base_name=str(zip_path.with_suffix("")),
    format="zip",
    root_dir=str(project_dir.parent),
    base_dir=project_dir.name
)

# Display ZIP information
print("=" * 60)
print("PROJECT ZIP CREATED SUCCESSFULLY")
print("=" * 60)

print("ZIP:", zip_path)
print("Size (MB):", round(zip_path.stat().st_size / (1024**2), 2))

# Automatically download
files.download(str(zip_path))

PROJECT ZIP CREATED SUCCESSFULLY
ZIP: /content/RAG_From_Scratch_Final.zip
Size (MB): 2.3


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## LLM Inference

In [28]:
# LOAD GENERATIVE LLM

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

LLM_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

device = "cuda" if torch.cuda.is_available() else "cpu"

print("=" * 65)
print("GENERATIVE LLM INITIALIZATION")
print("=" * 65)

print("Model:", LLM_NAME)
print("Device:", device)

# Keep the embedding model variable 'model' unchanged.
# Use separate variables for the generative LLM.

llm_tokenizer = AutoTokenizer.from_pretrained(
    LLM_NAME
)

llm_model = AutoModelForCausalLM.from_pretrained(
    LLM_NAME,
    torch_dtype=(
        torch.float16
        if device == "cuda"
        else torch.float32
    )
)

llm_model = llm_model.to(device)
llm_model.eval()

total_parameters = sum(
    p.numel()
    for p in llm_model.parameters()
)

print("\nModel loaded successfully!")
print(f"Parameters: {total_parameters / 1e6:.2f} million")
print("Device:", next(llm_model.parameters()).device)
print("Model status: READY")

GENERATIVE LLM INITIALIZATION
Model: Qwen/Qwen2.5-0.5B-Instruct
Device: cuda


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

... (2 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]


Model loaded successfully!
Parameters: 494.03 million
Device: cuda:0
Model status: READY

STEP 20 COMPLETE


In [29]:
# GENERATIVE RAG INFERENCE

import torch
import re

def generate_rag_answer(
    question,
    chunk_set="small",
    top_k=5,
    max_new_tokens=180
):
    """
    End-to-end generative RAG:

    Question
      -> Hybrid retrieval
      -> Grounded prompt
      -> Qwen inference
      -> Answer with PDF citations
    """

    # --------------------------------------------------------
    # 1. Retrieve supporting passages
    # --------------------------------------------------------

    retrieved = hybrid_retrieve(
        query=question,
        chunk_set=chunk_set,
        top_k=top_k,
        candidate_k=30,
        rrf_k=60
    )

    # --------------------------------------------------------
    # 2. Prepare evidence with citation identifiers
    # --------------------------------------------------------

    context_parts = []

    for i, item in enumerate(retrieved, start=1):

        context_parts.append(
            f"[S{i}] PDF page {item['page']}\n"
            f"{item['text']}"
        )

    context = "\n\n".join(context_parts)

    # --------------------------------------------------------
    # 3. Construct grounded prompt
    # --------------------------------------------------------

    system_prompt = (
        "You are a document-grounded question-answering assistant. "
        "Answer using ONLY the supplied source passages. "
        "Do not use outside knowledge. "
        "Do not invent facts, definitions, or references. "
        "Keep your answer concise and directly address the question. "
        "Cite supporting passages using [S1], [S2], etc. "
        "Use only citation IDs present in the provided passages. "
        "If the passages do not contain enough information, "
        "reply exactly: "
        "'Insufficient information in the retrieved document.'"
    )

    user_prompt = (
        f"QUESTION:\n{question}\n\n"
        f"SOURCE PASSAGES:\n{context}\n\n"
        "Give a concise answer supported by the passages. "
        "Include source citations."
    )

    messages = [
        {
            "role": "system",
            "content": system_prompt
        },
        {
            "role": "user",
            "content": user_prompt
        }
    ]

    # --------------------------------------------------------
    # 4. Apply Qwen chat template
    # --------------------------------------------------------

    prompt_text = llm_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = llm_tokenizer(
        prompt_text,
        return_tensors="pt",
        truncation=True,
        max_length=6144
    ).to(device)

    # --------------------------------------------------------
    # 5. Run LLM inference
    # --------------------------------------------------------

    with torch.inference_mode():

        outputs = llm_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=llm_tokenizer.eos_token_id
        )

    input_length = inputs["input_ids"].shape[1]

    generated_tokens = outputs[0][input_length:]

    answer = llm_tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

    # --------------------------------------------------------
    # 6. Validate citation identifiers
    # --------------------------------------------------------

    valid_citations = {
        f"S{i}"
        for i in range(1, len(retrieved) + 1)
    }

    citations_used = set(
        re.findall(r"\[S(\d+)\]", answer)
    )

    invalid_citations = [
        f"S{number}"
        for number in citations_used
        if f"S{number}" not in valid_citations
    ]

    # --------------------------------------------------------
    # 7. Display answer and sources
    # --------------------------------------------------------

    print("=" * 80)
    print("QUESTION")
    print(question)

    print("\nGENERATED ANSWER")
    print(answer)

    print("\nRETRIEVED SOURCES")

    for i, item in enumerate(retrieved, start=1):
        print(
            f"[S{i}] PDF page {item['page']} | "
            f"Chunk {item['chunk_id']} | "
            f"RRF {item['score']:.5f}"
        )

    print("\nCITATION CHECK")

    if invalid_citations:
        print(
            "Invalid citation IDs:",
            invalid_citations
        )
    else:
        print("No out-of-range citation IDs detected.")

    return {
        "question": question,
        "answer": answer,
        "chunk_set": chunk_set,
        "sources": retrieved,
        "invalid_citations": invalid_citations
    }


# TEST 1 — KNOWN ANSWER

test_result = generate_rag_answer(
    question="What are the four core functions of the AI RMF?",
    chunk_set="small",
    top_k=5
)


QUESTION
What are the four core functions of the AI RMF?

GENERATED ANSWER
The four core functions of the AI RMF include:

1. Govern: This involves defining policies and guidelines for managing risks.
2. Map: It includes identifying potential risks and assessing them.
3. Measure: It involves quantifying the impact of identified risks.
4. Manage: It focuses on implementing corrective measures to mitigate risks.

These functions are organized into categories and subcategories within the AI RMF core.

RETRIEVED SOURCES
[S1] PDF page 25 | Chunk p25_t0 | RRF 0.03279
[S2] PDF page 25 | Chunk p25_t104 | RRF 0.03150
[S3] PDF page 44 | Chunk p44_t208 | RRF 0.03077
[S4] PDF page 8 | Chunk p8_t0 | RRF 0.03021
[S5] PDF page 37 | Chunk p37_t0 | RRF 0.02951

CITATION CHECK
No out-of-range citation IDs detected.


#### Improve grounding, citations, and abstention

In [30]:
# STRICTER GENERATIVE RAG

import re
import torch

ABSTAIN = "Insufficient information in the retrieved document."


def generate_rag_answer_v2(
    question,
    chunk_set="small",
    top_k=5,
    max_new_tokens=180
):
    # 1. Retrieve evidence
    retrieved = hybrid_retrieve(
        query=question,
        chunk_set=chunk_set,
        top_k=top_k,
        candidate_k=30,
        rrf_k=60
    )

    # 2. Build context
    context_parts = []

    for i, item in enumerate(retrieved, start=1):
        context_parts.append(
            f"[S{i}] PDF page {item['page']}\n"
            f"{item['text']}"
        )

    context = "\n\n".join(context_parts)

    # 3. Grounded instructions
    system_prompt = f"""
You are a precise document question-answering assistant.

Rules:
1. Answer ONLY using the supplied source passages.
2. Answer the exact question without adding unrelated details.
3. Keep the answer short, usually 1-3 sentences.
4. Every factual answer MUST include at least one citation
   in the format [S1], [S2], etc.
5. Cite only passages that actually support the statement.
6. Do not invent citations or unsupported explanations.
7. If the passages do not adequately answer the question,
   reply exactly:
   {ABSTAIN}
8. If abstaining, do not add citations.

Example:
Question: What are the four functions?
Answer: The four functions are Govern, Map, Measure,
and Manage. [S1]
""".strip()

    user_prompt = f"""
Question:
{question}

Retrieved source passages:
{context}

Write the answer with supporting source citations.
""".strip()

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]

    prompt = llm_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = llm_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=6144
    ).to(next(llm_model.parameters()).device)

    # 4. Generate
    with torch.inference_mode():
        outputs = llm_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=llm_tokenizer.eos_token_id
        )

    new_tokens = outputs[0][
        inputs["input_ids"].shape[1]:
    ]

    answer = llm_tokenizer.decode(
        new_tokens,
        skip_special_tokens=True
    ).strip()

    # 5. Validate citation identifiers
    cited_numbers = re.findall(
        r"\[S(\d+)\]",
        answer
    )

    valid_ids = {
        str(i)
        for i in range(1, len(retrieved) + 1)
    }

    invalid_citations = [
        f"S{n}"
        for n in cited_numbers
        if n not in valid_ids
    ]

    abstained = (
        answer.strip().rstrip(".").lower()
        == ABSTAIN.rstrip(".").lower()
    )

    has_citation = len(cited_numbers) > 0

    citation_format_pass = (
        abstained
        or (
            has_citation
            and not invalid_citations
        )
    )

    # This checks citation presence and IDs,
    # not factual support or answer accuracy.

    print("=" * 75)
    print("QUESTION:", question)

    print("\nGENERATED ANSWER:")
    print(answer)

    print("\nCITATION FORMAT CHECK:")
    print("Abstained:", abstained)
    print("Citation present:", has_citation)
    print("Invalid citation IDs:", invalid_citations)
    print("Format valid:", citation_format_pass)

    print("\nAVAILABLE SOURCES:")
    for i, item in enumerate(retrieved, start=1):
        print(
            f"[S{i}] PDF page {item['page']} | "
            f"Chunk {item['chunk_id']}"
        )

    return {
        "question": question,
        "answer": answer,
        "abstained": abstained,
        "citation_present": has_citation,
        "invalid_citations": invalid_citations,
        "citation_format_pass": citation_format_pass,
        "sources": retrieved
    }


# ============================================================
# TEST A — ANSWERABLE
# ============================================================

answerable_result = generate_rag_answer_v2(
    "What are the four core functions of the AI RMF?"
)

# ============================================================
# TEST B — OUT-OF-SCOPE
# ============================================================

unanswerable_result = generate_rag_answer_v2(
    "What was the closing stock price of NVIDIA on January 5, 2024?"
)

print("\nSTEP 22 COMPLETE")

QUESTION: What are the four core functions of the AI RMF?

GENERATED ANSWER:
The four core functions of the AI RMF are govern, map, measure, and manage. [S1]

CITATION FORMAT CHECK:
Abstained: False
Citation present: True
Invalid citation IDs: []
Format valid: True

AVAILABLE SOURCES:
[S1] PDF page 25 | Chunk p25_t0
[S2] PDF page 25 | Chunk p25_t104
[S3] PDF page 44 | Chunk p44_t208
[S4] PDF page 8 | Chunk p8_t0
[S5] PDF page 37 | Chunk p37_t0
QUESTION: What was the closing stock price of NVIDIA on January 5, 2024?

GENERATED ANSWER:
The closing stock price of NVIDIA on January 5, 2024 was $12.50 per share. This information can be found in the following passage:

[PDF page 2, NIST AI 100 - 1 AI RMF 1. 0 Risks to Interpretability often can be addressed by communicating a description of why an AI system made a particular prediction or recommendation. Transparency, explainability, and interpretability are distinct characteristics that support each other. Transparency can answer the questi

#### Add evidence gating and strict citation validation

In [31]:
# EVIDENCE-GATED GENERATIVE RAG

import re
import json
from pathlib import Path

ABSTAIN = "Insufficient information in the retrieved document."

STOPWORDS = {
    "what", "which", "when", "where", "why", "how",
    "does", "do", "did", "is", "are", "was", "were",
    "the", "a", "an", "of", "in", "on", "for", "to",
    "and", "or", "with", "from", "by", "about",
    "can", "should", "would", "could", "its",
    "this", "that", "these", "those", "it",
    "be", "been", "being", "as", "at", "their"
}


def important_terms(text):
    words = re.findall(
        r"[a-z0-9]+",
        text.lower()
    )

    return set(
        word
        for word in words
        if len(word) >= 3
        and word not in STOPWORDS
    )


def evidence_relevance_check(
    question,
    retrieved,
    min_coverage=0.45
):
    """
    Check whether any retrieved passage contains
    enough important query terms.

    This is a lexical heuristic, not a semantic
    entailment or factual-grounding verifier.
    """

    query_terms = important_terms(question)

    if not query_terms:
        return {
            "relevant": False,
            "coverage": 0.0,
            "matched_terms": [],
            "query_terms": []
        }

    best_coverage = 0.0
    best_matches = set()

    for item in retrieved:

        passage_terms = important_terms(
            item["text"]
        )

        matched = query_terms & passage_terms

        coverage = (
            len(matched) / len(query_terms)
        )

        if coverage > best_coverage:
            best_coverage = coverage
            best_matches = matched

    return {
        "relevant": best_coverage >= min_coverage,
        "coverage": round(best_coverage, 4),
        "matched_terms": sorted(best_matches),
        "query_terms": sorted(query_terms)
    }


def generate_rag_answer_v3(
    question,
    chunk_set="small",
    top_k=5,
    min_coverage=0.45
):
    # --------------------------------------------------------
    # 1. Hybrid retrieval
    # --------------------------------------------------------

    retrieved = hybrid_retrieve(
        query=question,
        chunk_set=chunk_set,
        top_k=top_k,
        candidate_k=30,
        rrf_k=60
    )

    # --------------------------------------------------------
    # 2. Evidence relevance gate
    # --------------------------------------------------------

    relevance = evidence_relevance_check(
        question,
        retrieved,
        min_coverage=min_coverage
    )

    if not relevance["relevant"]:

        result = {
            "question": question,
            "answer": ABSTAIN,
            "raw_llm_answer": None,
            "abstained": True,
            "abstention_reason": "evidence_gate",
            "citation_format_pass": True,
            "relevance": relevance,
            "sources": retrieved
        }

    else:

        # ----------------------------------------------------
        # 3. Actual LLM inference using existing v2
        # ----------------------------------------------------

        llm_result = generate_rag_answer_v2(
            question=question,
            chunk_set=chunk_set,
            top_k=top_k
        )

        raw_answer = llm_result["answer"]

        valid_format = (
            llm_result["citation_format_pass"]
        )

        # If the model produces no valid citation,
        # do not present its text as a grounded answer.

        if not valid_format:

            final_answer = ABSTAIN
            abstained = True
            reason = "invalid_or_missing_citation"

        else:

            final_answer = raw_answer
            abstained = llm_result["abstained"]
            reason = (
                "llm_abstention"
                if abstained
                else None
            )

        result = {
            "question": question,
            "answer": final_answer,
            "raw_llm_answer": raw_answer,
            "abstained": abstained,
            "abstention_reason": reason,
            "citation_format_pass": valid_format,
            "relevance": relevance,
            "sources": retrieved
        }

    # --------------------------------------------------------
    # 4. Display result
    # --------------------------------------------------------

    print("\n" + "=" * 75)
    print("EVIDENCE-GATED RAG RESULT")
    print("=" * 75)

    print("Question:", question)
    print("Relevant evidence:", relevance["relevant"])
    print("Lexical coverage:", relevance["coverage"])

    print(
        "Matched terms:",
        relevance["matched_terms"]
    )

    print("\nFINAL ANSWER:")
    print(result["answer"])

    print("\nAbstained:", result["abstained"])
    print(
        "Abstention reason:",
        result["abstention_reason"]
    )

    return result


# ============================================================
# 5. TEST THE SAME TWO QUESTIONS
# ============================================================

print("\nTEST A — ANSWERABLE QUESTION")

test_answerable_v3 = generate_rag_answer_v3(
    "What are the four core functions of the AI RMF?"
)

print("\nTEST B — OUT-OF-SCOPE QUESTION")

test_unanswerable_v3 = generate_rag_answer_v3(
    "What was the closing stock price of NVIDIA on January 5, 2024?"
)

print("\n" + "=" * 75)
print("STEP 23 TEST SUMMARY")
print("=" * 75)

print(
    "Answerable question produced an answer:",
    not test_answerable_v3["abstained"]
)

print(
    "Out-of-scope question abstained:",
    test_unanswerable_v3["abstained"]
)



TEST A — ANSWERABLE QUESTION
QUESTION: What are the four core functions of the AI RMF?

GENERATED ANSWER:
The four core functions of the AI RMF are govern, map, measure, and manage. [S1]

CITATION FORMAT CHECK:
Abstained: False
Citation present: True
Invalid citation IDs: []
Format valid: True

AVAILABLE SOURCES:
[S1] PDF page 25 | Chunk p25_t0
[S2] PDF page 25 | Chunk p25_t104
[S3] PDF page 44 | Chunk p44_t208
[S4] PDF page 8 | Chunk p8_t0
[S5] PDF page 37 | Chunk p37_t0

EVIDENCE-GATED RAG RESULT
Question: What are the four core functions of the AI RMF?
Relevant evidence: True
Lexical coverage: 1.0
Matched terms: ['core', 'four', 'functions', 'rmf']

FINAL ANSWER:
The four core functions of the AI RMF are govern, map, measure, and manage. [S1]

Abstained: False
Abstention reason: None

TEST B — OUT-OF-SCOPE QUESTION

EVIDENCE-GATED RAG RESULT
Question: What was the closing stock price of NVIDIA on January 5, 2024?
Relevant evidence: False
Lexical coverage: 0.1667
Matched terms: ['ja

#### Evaluate generative RAG on all 15 questions

In [32]:
# FULL GENERATIVE RAG EVALUATION

import re
import json
import io
import contextlib
import pandas as pd
from pathlib import Path

BASE = Path("/content/rag_from_scratch")
EVAL_DIR = BASE / "evaluation"
EVAL_DIR.mkdir(parents=True, exist_ok=True)

questions_df = pd.read_csv(
    EVAL_DIR / "questions_15.csv"
)

assert len(questions_df) == 15

generation_records = []

print("=" * 80)
print("GENERATIVE RAG EVALUATION — 15 QUESTIONS")
print("=" * 80)

for _, row in questions_df.iterrows():

    qid = str(row["question_id"])
    question = str(row["question"])

    # Suppress detailed internal prints; keep results.
    with contextlib.redirect_stdout(io.StringIO()):
        result = generate_rag_answer_v3(
            question=question,
            chunk_set="small",
            top_k=5,
            min_coverage=0.45
        )

    answer = result["answer"]
    sources = result["sources"]

    # Extract citations used in answer.
    cited_ids = [
        int(n)
        for n in re.findall(r"\[S(\d+)\]", answer)
    ]

    citation_pages = sorted(set(
        int(sources[n - 1]["page"])
        for n in cited_ids
        if 1 <= n <= len(sources)
    ))

    generation_records.append({
        "question_id": qid,
        "question": question,
        "gold_page": int(row["gold_page"]),
        "generated_answer": answer,
        "abstained": bool(result["abstained"]),
        "abstention_reason": result["abstention_reason"],
        "lexical_coverage": result["relevance"]["coverage"],
        "citation_format_pass": bool(
            result["citation_format_pass"]
        ),
        "cited_source_ids": [
            f"S{n}" for n in cited_ids
        ],
        "cited_pdf_pages": citation_pages,
        "gold_page_cited": (
            int(row["gold_page"]) in citation_pages
        ),
        "manual_answer_correct": None,
        "manual_citation_supported": None,
        "review_notes": ""
    })

    print("\n" + "-" * 80)
    print(f"{qid}: {question}")
    print("Answer:", answer)
    print("Abstained:", result["abstained"])
    print("Citation format valid:", result["citation_format_pass"])
    print("Cited pages:", citation_pages)


# ============================================================
# SAVE RESULTS
# ============================================================

json_path = EVAL_DIR / "generative_rag_15.json"
csv_path = EVAL_DIR / "generative_rag_15.csv"

with open(json_path, "w", encoding="utf-8") as f:
    json.dump(
        generation_records,
        f,
        indent=2,
        ensure_ascii=False
    )

csv_records = []

for record in generation_records:

    item = record.copy()

    item["cited_source_ids"] = ", ".join(
        item["cited_source_ids"]
    )

    item["cited_pdf_pages"] = ", ".join(
        map(str, item["cited_pdf_pages"])
    )

    csv_records.append(item)

pd.DataFrame(csv_records).to_csv(
    csv_path,
    index=False
)


# ============================================================
# AUTOMATIC SUMMARY — NOT FACTUAL ACCURACY
# ============================================================

total = len(generation_records)

answered = sum(
    not r["abstained"]
    for r in generation_records
)

abstained = total - answered

valid_citations = sum(
    r["citation_format_pass"]
    for r in generation_records
)

gold_page_cited = sum(
    r["gold_page_cited"]
    for r in generation_records
)

print("\n" + "=" * 80)
print("GENERATIVE RAG EVALUATION SUMMARY")
print("=" * 80)

print("Questions:", total)
print("Answered:", answered)
print("Abstained:", abstained)
print("Citation format passed:", valid_citations)
print("Gold page cited:", gold_page_cited)
print("Manually verified answers: 0 /", total)

print("\nSaved:")
print(json_path)
print(csv_path)


GENERATIVE RAG EVALUATION — 15 QUESTIONS

--------------------------------------------------------------------------------
Q01: What are the four core functions of the AI RMF?
Answer: The four core functions of the AI RMF are govern, map, measure, and manage. [S1]
Abstained: False
Citation format valid: True
Cited pages: [25]

--------------------------------------------------------------------------------
Q02: What is the purpose of the GOVERN function?
Answer: The purpose of the GOVERN function is to provide organizations with the opportunity to clarify and define the roles and responsibilities for the humans in the human-AI team configurations and those who are overseeing the AI system performance. This helps ensure that the AI system operates effectively and efficiently while maintaining ethical standards and avoiding potential risks. [S1]
Abstained: False
Citation format valid: True
Cited pages: [27]

--------------------------------------------------------------------------------

In [33]:
# GENERATIVE RAG FAILURE DIAGNOSIS

import json
from pathlib import Path

BASE = Path("/content/rag_from_scratch")
EVAL_DIR = BASE / "evaluation"

with open(
    EVAL_DIR / "generative_rag_15.json",
    encoding="utf-8"
) as f:
    records = json.load(f)

with open(
    EVAL_DIR / "questions_15.csv",
    encoding="utf-8"
) as f:
    import pandas as pd
    questions_df = pd.read_csv(f)

question_lookup = {
    str(row["question_id"]): str(row["question"])
    for _, row in questions_df.iterrows()
}

print("=" * 80)
print("ABSTENTION FAILURE DIAGNOSIS")
print("=" * 80)

for record in records:

    if not record["abstained"]:
        continue

    qid = record["question_id"]
    question = question_lookup[qid]

    # Reproduce retrieval and gate only.
    # No generative inference.
    retrieved = hybrid_retrieve(
        query=question,
        chunk_set="small",
        top_k=5,
        candidate_k=30,
        rrf_k=60
    )

    relevance = evidence_relevance_check(
        question,
        retrieved,
        min_coverage=0.45
    )

    reason = record["abstention_reason"]

    print("\n" + "-" * 80)
    print(qid, "-", question)
    print("-" * 80)

    print("Abstention reason:", reason)
    print("Lexical coverage:", relevance["coverage"])
    print("Gate passed:", relevance["relevant"])
    print("Matched query terms:", relevance["matched_terms"])
    print("All query terms:", relevance["query_terms"])

    print("\nRetrieved source pages:")

    for item in retrieved:
        print(
            f"  Rank {item['rank']}: "
            f"PDF page {item['page']} | "
            f"Chunk {item['chunk_id']}"
        )

    print("\nDiagnosis:")

    if reason == "evidence_gate":
        print(
            "The lexical relevance gate rejected the "
            "question before LLM inference."
        )

    elif reason == "invalid_or_missing_citation":
        print(
            "The question passed the evidence gate, "
            "but the LLM output failed citation validation."
        )

    elif reason == "llm_abstention":
        print(
            "The question passed the gate, "
            "but the LLM chose to abstain."
        )

    else:
        print(
            "Unexpected or unknown abstention reason:",
            reason
        )


print("\n" + "=" * 80)
print("ABSTENTION COUNTS BY REASON")
print("=" * 80)

reason_counts = {}

for record in records:

    if record["abstained"]:

        reason = (
            record["abstention_reason"]
            or "unknown"
        )

        reason_counts[reason] = (
            reason_counts.get(reason, 0) + 1
        )

for reason, count in sorted(reason_counts.items()):
    print(f"{reason}: {count}")


ABSTENTION FAILURE DIAGNOSIS

--------------------------------------------------------------------------------
Q03 - What does the MAP function establish about an AI system?
--------------------------------------------------------------------------------
Abstention reason: invalid_or_missing_citation
Lexical coverage: 0.75
Gate passed: True
Matched query terms: ['function', 'map', 'system']
All query terms: ['establish', 'function', 'map', 'system']

Retrieved source pages:
  Rank 1: PDF page 30 | Chunk p30_t312
  Rank 2: PDF page 31 | Chunk p31_t0
  Rank 3: PDF page 29 | Chunk p29_t312
  Rank 4: PDF page 30 | Chunk p30_t0
  Rank 5: PDF page 46 | Chunk p46_t104

Diagnosis:
The question passed the evidence gate, but the LLM output failed citation validation.

--------------------------------------------------------------------------------
Q04 - What is the purpose of the MEASURE function?
--------------------------------------------------------------------------------
Abstention reason:

In [34]:
# INSPECT RAW LLM FAILURE OUTPUTS

import re

diagnostic_questions = {
    "Q03": "What does the MAP function establish about an AI system?",
    "Q04": "What is the purpose of the MEASURE function?",
    "Q15": "Why is AI risk management considered a continuous process?"
}

raw_diagnostics = {}

for qid, question in diagnostic_questions.items():

    print("\n" + "=" * 85)
    print(f"RAW LLM DIAGNOSIS — {qid}")
    print("=" * 85)

    result = generate_rag_answer_v2(
        question=question,
        chunk_set="small",
        top_k=5,
        max_new_tokens=256
    )

    answer = result["answer"]

    citations = re.findall(
        r"\[S(\d+)\]",
        answer
    )

    raw_diagnostics[qid] = {
        "question": question,
        "raw_answer": answer,
        "citation_ids": citations,
        "citation_format_pass": result["citation_format_pass"]
    }

    print("\n" + "-" * 70)
    print("DIAGNOSTIC SUMMARY")
    print("-" * 70)

    print("Answer length:", len(answer))
    print("Citation IDs:", citations)
    print(
        "Citation format valid:",
        result["citation_format_pass"]
    )

    print("\nTOP 3 RETRIEVED PASSAGE PREVIEWS:")

    for i, item in enumerate(
        result["sources"][:3],
        start=1
    ):
        print(
            f"\n[S{i}] PDF page {item['page']} "
            f"| Chunk {item['chunk_id']}"
        )
        print(item["text"][:650])




RAW LLM DIAGNOSIS — Q03
QUESTION: What does the MAP function establish about an AI system?

GENERATED ANSWER:
The MAP function establishes the context to frame risks related to an AI system by understanding the intended purposes, potential impacts on individuals, communities, organizations, society, and the planet. It also considers the impact of ai systems on users, including their expectations and potential benefits. This helps in developing effective risk management strategies and mitigating uncertainties. Without comprehensive contextual knowledge, organizations can identify risks and take appropriate actions.

CITATION FORMAT CHECK:
Abstained: False
Citation present: False
Invalid citation IDs: []
Format valid: False

AVAILABLE SOURCES:
[S1] PDF page 30 | Chunk p30_t312
[S2] PDF page 31 | Chunk p31_t0
[S3] PDF page 29 | Chunk p29_t312
[S4] PDF page 30 | Chunk p30_t0
[S5] PDF page 46 | Chunk p46_t104

----------------------------------------------------------------------
DIAGNOSTI

#### Citation-aware LLM inference

In [35]:
# CITATION-AWARE GENERATIVE RAG

import re
import torch

ABSTAIN = "Insufficient information in the retrieved document."


def generate_rag_answer_v4(
    question,
    chunk_set="small",
    top_k=5,
    min_coverage=0.45,
    max_new_tokens=160
):

    # 1. Retrieve evidence
    sources = hybrid_retrieve(
        query=question,
        chunk_set=chunk_set,
        top_k=top_k,
        candidate_k=30,
        rrf_k=60
    )

    # 2. Check evidence relevance
    relevance = evidence_relevance_check(
        question,
        sources,
        min_coverage=min_coverage
    )

    result = {
        "question": question,
        "answer": ABSTAIN,
        "abstained": True,
        "abstention_reason": None,
        "sources": sources,
        "relevance": relevance,
        "raw_attempts": [],
        "citation_format_pass": False
    }

    if not relevance["relevant"]:
        result["abstention_reason"] = "evidence_gate"
        return result

    # 3. Prepare cited source passages
    context = "\n\n".join(
        f"[S{i}] PDF page {item['page']}\n"
        f"{item['text']}"
        for i, item in enumerate(sources, 1)
    )

    def run_generation(messages):

        prompt = llm_tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        inputs = llm_tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=6144
        ).to(next(llm_model.parameters()).device)

        with torch.inference_mode():

            outputs = llm_model.generate(
                **inputs,
                do_sample=False,
                max_new_tokens=max_new_tokens,
                pad_token_id=llm_tokenizer.eos_token_id
            )

        new_tokens = outputs[0][
            inputs["input_ids"].shape[1]:
        ]

        return llm_tokenizer.decode(
            new_tokens,
            skip_special_tokens=True
        ).strip()

    # 4. Generate a concise answer
    system_message = (
        "You answer questions using only supplied document excerpts. "
        "Use one or two short sentences. "
        "Include a citation [S1], [S2], etc. immediately after "
        "each factual claim. "
        "Only cite passages that support the claim. "
        "Never use outside knowledge. "
        "If no passage supports an answer, respond exactly: "
        + ABSTAIN
    )

    user_message = (
        f"Question:\n{question}\n\n"
        f"Document excerpts:\n{context}\n\n"
        "Answer the question directly and cite the evidence. "
        "Do not describe the retrieval process or list the sources."
    )

    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": user_message}
    ]

    def validate(answer):

        if answer.strip() == ABSTAIN:
            return True, True, []

        citation_numbers = re.findall(
            r"\[S(\d+)\]",
            answer
        )

        invalid = [
            int(n)
            for n in citation_numbers
            if not (1 <= int(n) <= len(sources))
        ]

        valid_format = (
            len(citation_numbers) > 0
            and not invalid
        )

        return valid_format, False, citation_numbers

    first_answer = run_generation(messages)
    result["raw_attempts"].append(first_answer)

    valid, abstained, citations = validate(first_answer)

    # 5. Retry once if citations are missing or invalid
    if not valid:

        repair_messages = [
            {"role": "system", "content": system_message},
            {
                "role": "user",
                "content": (
                    user_message
                    + "\n\nYour previous response was:\n"
                    + first_answer
                    + "\n\nRewrite it in one or two sentences. "
                    "Use only facts supported by the excerpts. "
                    "Every factual claim must have a citation "
                    "such as [S1] or [S3]. "
                    "If unsupported, output the exact "
                    "insufficient-information response."
                )
            }
        ]

        second_answer = run_generation(repair_messages)

        result["raw_attempts"].append(second_answer)

        valid, abstained, citations = validate(second_answer)

        final_candidate = second_answer

    else:
        final_candidate = first_answer

    # 6. Enforce citation formatting
    if valid:

        result["answer"] = final_candidate
        result["abstained"] = abstained
        result["citation_format_pass"] = True

        if abstained:
            result["abstention_reason"] = "llm_abstention"

    else:

        result["answer"] = ABSTAIN
        result["abstained"] = True
        result["abstention_reason"] = (
            "citation_validation_failed"
        )

    return result


# ============================================================
# TEST THE THREE PREVIOUS FAILURES
# ============================================================

test_questions = {
    "Q03": "What does the MAP function establish about an AI system?",
    "Q04": "What is the purpose of the MEASURE function?",
    "Q15": "Why is AI risk management considered a continuous process?"
}

v4_test_results = {}

for qid, question in test_questions.items():

    result = generate_rag_answer_v4(question)

    v4_test_results[qid] = result

    print("\n" + "=" * 80)
    print(qid, "-", question)
    print("=" * 80)

    print("Final answer:")
    print(result["answer"])

    print("\nAbstained:", result["abstained"])
    print("Citation format passed:", result["citation_format_pass"])
    print("Generation attempts:", len(result["raw_attempts"]))

    print("\nCited source mapping:")

    for number in sorted(set(
        int(n)
        for n in re.findall(
            r"\[S(\d+)\]",
            result["answer"]
        )
    )):
        if 1 <= number <= len(result["sources"]):

            source = result["sources"][number - 1]

            print(
                f"[S{number}] PDF page {source['page']} "
                f"| Chunk {source['chunk_id']}"
            )

    if result["abstained"]:
        print(
            "\nAbstention reason:",
            result["abstention_reason"]
        )



Q03 - What does the MAP function establish about an AI system?
Final answer:
Insufficient information in the retrieved document.

Abstained: True
Citation format passed: False
Generation attempts: 2

Cited source mapping:

Abstention reason: citation_validation_failed

Q04 - What is the purpose of the MEASURE function?
Final answer:
Insufficient information in the retrieved document.

Abstained: True
Citation format passed: False
Generation attempts: 2

Cited source mapping:

Abstention reason: citation_validation_failed

Q15 - Why is AI risk management considered a continuous process?
Final answer:
Insufficient information in the retrieved document.

Abstained: True
Citation format passed: False
Generation attempts: 2

Cited source mapping:

Abstention reason: citation_validation_failed


In [36]:
# ============================================================
# STEP 28 — INSPECT FAILED GENERATION ATTEMPTS
# ============================================================

import re

print("=" * 80)
print("RAW GENERATION ATTEMPT AUDIT")
print("=" * 80)

for qid, result in v4_test_results.items():

    print("\n" + "=" * 80)
    print("QUESTION:", qid)
    print(result["question"])
    print("=" * 80)

    for attempt_number, answer in enumerate(
        result["raw_attempts"],
        start=1
    ):

        citations = re.findall(
            r"\[S(\d+)\]",
            answer
        )

        print(f"\n--- ATTEMPT {attempt_number} ---")

        print("Characters:", len(answer))
        print("Citation IDs:", citations)

        print("\nRAW ANSWER:")
        print(repr(answer))

        print("\nReadable answer:")
        print(answer)

    print("\nRETRIEVED SOURCES:")

    for i, source in enumerate(
        result["sources"],
        start=1
    ):
        print(
            f"[S{i}] PDF page {source['page']} "
            f"| {source['chunk_id']}"
        )

print("\nSTEP 28 COMPLETE")

RAW GENERATION ATTEMPT AUDIT

QUESTION: Q03
What does the MAP function establish about an AI system?

--- ATTEMPT 1 ---
Characters: 80
Citation IDs: []

RAW ANSWER:
'The MAP function establishes the context to frame risks related to an AI system.'

Readable answer:
The MAP function establishes the context to frame risks related to an AI system.

--- ATTEMPT 2 ---
Characters: 80
Citation IDs: []

RAW ANSWER:
'The MAP function establishes the context to frame risks related to an AI system.'

Readable answer:
The MAP function establishes the context to frame risks related to an AI system.

RETRIEVED SOURCES:
[S1] PDF page 30 | p30_t312
[S2] PDF page 31 | p31_t0
[S3] PDF page 29 | p29_t312
[S4] PDF page 30 | p30_t0
[S5] PDF page 46 | p46_t104

QUESTION: Q04
What is the purpose of the MEASURE function?

--- ATTEMPT 1 ---
Characters: 200
Citation IDs: []

RAW ANSWER:
'The purpose of the MEASURE function is to provide unique and meaningful information to assess AI risks, including privacy ris

Qwen is producing useful answers but repeatedly ignoring the citation format.

## Evidence-verified citation generation

In [37]:
import json
import re
import torch

ABSTAIN = "Insufficient information in the retrieved document."


def normalize_evidence(text):
    """Normalize whitespace and punctuation for quote matching."""
    text = text.lower()
    text = re.sub(r"\s+", " ", text)
    return re.findall(r"[a-z0-9]+", text)


def verify_quote(quote, source_text, min_words=8):
    """
    Verify that a quoted passage actually exists
    as consecutive words in the selected source.
    """
    quote_words = normalize_evidence(quote)
    source_words = normalize_evidence(source_text)

    if len(quote_words) < min_words:
        return False

    n = len(quote_words)

    return any(
        source_words[i:i+n] == quote_words
        for i in range(len(source_words) - n + 1)
    )


def parse_llm_json(text):
    """Parse a JSON object, allowing optional markdown fences."""

    cleaned = text.strip()

    cleaned = re.sub(
        r"^```(?:json)?\s*",
        "",
        cleaned,
        flags=re.IGNORECASE
    )

    cleaned = re.sub(r"\s*```$", "", cleaned)

    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        pass

    # Try extracting a complete JSON object.
    start = cleaned.find("{")
    end = cleaned.rfind("}")

    if start >= 0 and end > start:
        try:
            return json.loads(cleaned[start:end+1])
        except json.JSONDecodeError:
            pass

    return None


def generate_rag_answer_v5(
    question,
    chunk_set="small",
    top_k=5,
    min_coverage=0.45
):

    sources = hybrid_retrieve(
        question,
        chunk_set=chunk_set,
        top_k=top_k,
        candidate_k=30,
        rrf_k=60
    )

    relevance = evidence_relevance_check(
        question,
        sources,
        min_coverage=min_coverage
    )

    result = {
        "question": question,
        "answer": ABSTAIN,
        "abstained": True,
        "reason": None,
        "citation_verified": False,
        "source_id": None,
        "source_page": None,
        "evidence_quote": None,
        "raw_output": None,
        "sources": sources
    }

    if not relevance["relevant"]:
        result["reason"] = "evidence_gate"
        return result

    context = "\n\n".join(
        f"S{i} (PDF page {item['page']}):\n"
        f"{item['text']}"
        for i, item in enumerate(sources, 1)
    )

    system_prompt = """
You answer questions using only provided document passages.

Return exactly one JSON object with these fields:
{
  "answer": "A short factual answer.",
  "source_id": "S1",
  "evidence_quote": "An exact consecutive quote from the source."
}

Rules:
- Keep the answer to one or two sentences.
- Select the passage that most directly supports the answer.
- Copy at least 8 consecutive words exactly from that passage.
- Do not change wording in the evidence_quote.
- Do not invent facts or evidence.
- Use a source_id from the provided passages.
- Do not include markdown or explanations outside the JSON.
- If the passages do not support an answer, use:
  {"answer": "Insufficient information in the retrieved document.",
   "source_id": null,
   "evidence_quote": null}
""".strip()

    user_prompt = (
        f"Question:\n{question}\n\n"
        f"Document passages:\n{context}\n\n"
        "Return the required JSON object."
    )

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]

    prompt = llm_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = llm_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=6144
    ).to(next(llm_model.parameters()).device)

    with torch.inference_mode():
        output = llm_model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
            pad_token_id=llm_tokenizer.eos_token_id
        )

    generated = output[0][
        inputs["input_ids"].shape[1]:
    ]

    raw_output = llm_tokenizer.decode(
        generated,
        skip_special_tokens=True
    ).strip()

    result["raw_output"] = raw_output

    data = parse_llm_json(raw_output)

    if not isinstance(data, dict):
        result["reason"] = "invalid_json"
        return result

    answer = data.get("answer")
    source_id = data.get("source_id")
    quote = data.get("evidence_quote")

    if not isinstance(answer, str) or not answer.strip():
        result["reason"] = "missing_answer"
        return result

    if answer.strip() == ABSTAIN:
        result["reason"] = "llm_abstention"
        return result

    if not isinstance(source_id, str):
        result["reason"] = "missing_source_id"
        return result

    if not isinstance(quote, str):
        result["reason"] = "missing_evidence_quote"
        return result

    match = re.fullmatch(r"S([1-9][0-9]*)", source_id)

    if not match:
        result["reason"] = "invalid_source_id"
        return result

    source_index = int(match.group(1)) - 1

    if not 0 <= source_index < len(sources):
        result["reason"] = "source_out_of_range"
        return result

    selected_source = sources[source_index]

    if not verify_quote(
        quote,
        selected_source["text"],
        min_words=8
    ):
        result["reason"] = "quote_not_verified"
        return result

    # The citation is constructed ONLY after
    # its quoted evidence has been verified.
    clean_answer = re.sub(
        r"\[S\d+\]",
        "",
        answer
    ).strip()

    result.update({
        "answer": f"{clean_answer} [{source_id}]",
        "abstained": False,
        "reason": None,
        "citation_verified": True,
        "source_id": source_id,
        "source_page": int(selected_source["page"]),
        "evidence_quote": quote
    })

    return result


# ============================================================
# TEST PREVIOUS FAILURES
# ============================================================

questions_to_test = {
    "Q03": "What does the MAP function establish about an AI system?",
    "Q04": "What is the purpose of the MEASURE function?",
    "Q15": "Why is AI risk management considered a continuous process?"
}

v5_test_results = {}

for qid, question in questions_to_test.items():

    result = generate_rag_answer_v5(question)

    v5_test_results[qid] = result

    print("\n" + "=" * 75)
    print(qid, "-", question)
    print("=" * 75)

    print("\nAnswer:")
    print(result["answer"])

    print("\nCitation verified:", result["citation_verified"])
    print("Source:", result["source_id"])
    print("PDF page:", result["source_page"])
    print("Abstention reason:", result["reason"])

    if result["evidence_quote"]:
        print("\nVerified supporting quote:")
        print(result["evidence_quote"])

    if not result["citation_verified"]:
        print("\nRaw model output:")
        print(result["raw_output"])



Q03 - What does the MAP function establish about an AI system?

Answer:
Insufficient information in the retrieved document.

Citation verified: False
Source: None
PDF page: None
Abstention reason: quote_not_verified

Raw model output:
{
  "answer": "The MAP function establishes the context to frame risks related to an AI system.",
  "source_id": "S1",
  "evidence_quote": "The MAP function establishes the context to frame risks related to an AI system."
}

Q04 - What is the purpose of the MEASURE function?

Answer:
The purpose of the MEASURE function is to provide unique and meaningful information to assess AI risks. [S1]

Citation verified: True
Source: S1
PDF page: 35
Abstention reason: None

Verified supporting quote:
measure 2. 6 : the ai system is evaluated regularly for safety risks – as identified in the map function. the ai system to be de - ployed is demonstrated to be safe, its residual negative risk does not exceed the risk tolerance, and it can fail safely, particularly if 

In [38]:
# QUESTION-FOCUSED EVIDENCE SELECTION

import re
import numpy as np


def split_evidence_sentences(text):
    """
    Split retrieved text into candidate evidence spans.

    Keep consecutive sentence pairs as well as individual
    sentences to reduce loss of context.
    """

    sentences = re.split(
        r"(?<=[.!?])\s+",
        text.strip()
    )

    sentences = [
        s.strip()
        for s in sentences
        if len(s.strip()) >= 45
    ]

    spans = []

    for i, sentence in enumerate(sentences):
        spans.append(sentence)

        if i + 1 < len(sentences):
            spans.append(
                sentence + " " + sentences[i + 1]
            )

    return spans


def select_question_evidence(
    question,
    chunk_set="small",
    candidate_k=15,
    top_n=5
):

    # Retrieve a larger candidate pool.
    retrieved = hybrid_retrieve(
        query=question,
        chunk_set=chunk_set,
        top_k=candidate_k,
        candidate_k=40,
        rrf_k=60
    )

    evidence = []
    seen = set()

    for item in retrieved:

        for span in split_evidence_sentences(
            item["text"]
        ):

            normalized = " ".join(
                span.lower().split()
            )

            if normalized in seen:
                continue

            seen.add(normalized)

            evidence.append({
                "text": span,
                "page": int(item["page"]),
                "chunk_id": item["chunk_id"],
                "retrieval_rank": int(item["rank"])
            })

    if not evidence:
        return []

    # Compare question and evidence semantically.
    query_vector = model.encode(
        [question],
        normalize_embeddings=True
    )[0]

    evidence_vectors = model.encode(
        [item["text"] for item in evidence],
        normalize_embeddings=True,
        batch_size=32
    )

    semantic_scores = (
        evidence_vectors @ query_vector
    )

    query_terms = important_terms(question)

    for i, item in enumerate(evidence):

        span_terms = important_terms(
            item["text"]
        )

        lexical_score = (
            len(query_terms & span_terms)
            / max(len(query_terms), 1)
        )

        # Prefer semantic relevance while also
        # rewarding exact terminology.
        item["score"] = float(
            0.75 * semantic_scores[i]
            + 0.25 * lexical_score
        )

    evidence.sort(
        key=lambda x: x["score"],
        reverse=True
    )

    # Avoid returning multiple overlapping spans
    # from the same chunk where possible.
    selected = []
    used_chunks = set()

    for item in evidence:

        if item["chunk_id"] in used_chunks:
            continue

        selected.append(item)
        used_chunks.add(item["chunk_id"])

        if len(selected) >= top_n:
            break

    return selected


# ============================================================
# TEST THREE PREVIOUS FAILURES
# ============================================================

test_questions = {
    "Q03": "What does the MAP function establish about an AI system?",
    "Q04": "What is the purpose of the MEASURE function?",
    "Q15": "Why is AI risk management considered a continuous process?"
}

for qid, question in test_questions.items():

    print("\n" + "=" * 85)
    print(qid, "-", question)
    print("=" * 85)

    selected = select_question_evidence(
        question,
        chunk_set="small",
        candidate_k=15,
        top_n=5
    )

    for i, item in enumerate(selected, start=1):

        print(
            f"\n[E{i}] PDF page {item['page']}"
            f" | Chunk {item['chunk_id']}"
            f" | Score {item['score']:.4f}"
        )

        print(item["text"])



Q03 - What does the MAP function establish about an AI system?

[E1] PDF page 29 | Chunk p29_t312 | Score 0.7778
the map function establishes the context to frame risks related to an ai system.

[E2] PDF page 29 | Chunk p29_t208 | Score 0.7620
2 map the map function establishes the context to frame risks related to an ai system.

[E3] PDF page 46 | Chunk p46_t104 | Score 0.7320
the map function suggests opportunities to define and document processes for operator and practitioner proficiency with ai system performance and trustworthiness concepts, and to define relevant technical standards and certifications.

[E4] PDF page 46 | Chunk p46_t0 | Score 0.7062
the map function suggests opportunities to define and document processes for operator and practitioner proficiency with ai system performance

[E5] PDF page 35 | Chunk p35_t104 | Score 0.7035
9 : the ai model is explained, validated, and docu - mented, and ai system output is interpreted within its context – as identified in the map 

In [39]:
# FOCUSED EVIDENCE + LLM INFERENCE

import torch
import re


ABSTAIN = "Insufficient information in the retrieved document."


def select_complete_evidence(
    question,
    chunk_set="small",
    candidate_k=15,
    top_n=4,
    min_words=12
):
    """
    Retrieve and select evidence while filtering out
    very short fragments.
    """

    # Request more evidence than needed so that
    # filtering does not leave too few candidates.
    candidates = select_question_evidence(
        question=question,
        chunk_set=chunk_set,
        candidate_k=candidate_k,
        top_n=15
    )

    selected = []

    for item in candidates:

        words = re.findall(
            r"\b\w+\b",
            item["text"]
        )

        if len(words) < min_words:
            continue

        selected.append(item)

        if len(selected) == top_n:
            break

    return selected


def focused_rag_generate(
    question,
    chunk_set="small",
    top_n=4,
    max_new_tokens=150
):

    # --------------------------------------------------------
    # 1. Select focused evidence
    # --------------------------------------------------------

    evidence = select_complete_evidence(
        question=question,
        chunk_set=chunk_set,
        candidate_k=15,
        top_n=top_n
    )

    if not evidence:

        return {
            "question": question,
            "answer": ABSTAIN,
            "abstained": True,
            "evidence": [],
            "raw_answer": "",
            "review_status": "pending"
        }

    # --------------------------------------------------------
    # 2. Construct compact evidence context
    # --------------------------------------------------------

    context = "\n\n".join(
        f"[E{i}] PDF page {item['page']}:\n"
        f"{item['text']}"
        for i, item in enumerate(evidence, start=1)
    )

    system_prompt = (
        "You answer questions using only the provided "
        "document evidence. "
        "Give a direct answer in one or two short sentences. "
        "Do not invent facts or examples. "
        "Do not include citations in your generated text; "
        "the application handles source references separately. "
        "If the evidence does not answer the question, "
        f"respond exactly: {ABSTAIN}"
    )

    user_prompt = (
        f"Question:\n{question}\n\n"
        f"Evidence:\n{context}\n\n"
        "Answer using only the evidence above."
    )

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]

    # --------------------------------------------------------
    # 3. LLM inference
    # --------------------------------------------------------

    prompt = llm_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = llm_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=4096
    ).to(next(llm_model.parameters()).device)

    with torch.inference_mode():

        outputs = llm_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=llm_tokenizer.eos_token_id
        )

    generated_tokens = outputs[0][
        inputs["input_ids"].shape[1]:
    ]

    raw_answer = llm_tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

    abstained = (
        raw_answer.rstrip(".").lower()
        == ABSTAIN.rstrip(".").lower()
    )

    # --------------------------------------------------------
    # 4. Return answer and explicit evidence provenance
    # --------------------------------------------------------

    return {
        "question": question,
        "answer": raw_answer,
        "abstained": abstained,
        "evidence": evidence,
        "raw_answer": raw_answer,
        "review_status": "pending"
    }


# ============================================================
# TEST Q03, Q04 AND Q15
# ============================================================

questions_to_test = {
    "Q03": "What does the MAP function establish about an AI system?",
    "Q04": "What is the purpose of the MEASURE function?",
    "Q15": "Why is AI risk management considered a continuous process?"
}

focused_test_results = {}

for qid, question in questions_to_test.items():

    result = focused_rag_generate(
        question,
        chunk_set="small",
        top_n=4
    )

    focused_test_results[qid] = result

    print("\n" + "=" * 80)
    print(qid, "-", question)
    print("=" * 80)

    print("\nGENERATED ANSWER:")
    print(result["answer"])

    print("\nABSTAINED:", result["abstained"])

    print("\nEVIDENCE SUPPLIED TO THE LLM:")

    for i, item in enumerate(
        result["evidence"],
        start=1
    ):

        print(
            f"\n[E{i}] PDF page {item['page']} "
            f"| Chunk {item['chunk_id']} "
            f"| Score {item['score']:.4f}"
        )

        print(item["text"])

    print("\nManual factual verification: PENDING")




Q03 - What does the MAP function establish about an AI system?

GENERATED ANSWER:
The MAP function establishes that risks are framed by the context of an AI system.

ABSTAINED: False

EVIDENCE SUPPLIED TO THE LLM:

[E1] PDF page 29 | Chunk p29_t312 | Score 0.7778
the map function establishes the context to frame risks related to an ai system.

[E2] PDF page 29 | Chunk p29_t208 | Score 0.7620
2 map the map function establishes the context to frame risks related to an ai system.

[E3] PDF page 46 | Chunk p46_t104 | Score 0.7320
the map function suggests opportunities to define and document processes for operator and practitioner proficiency with ai system performance and trustworthiness concepts, and to define relevant technical standards and certifications.

[E4] PDF page 46 | Chunk p46_t0 | Score 0.7062
the map function suggests opportunities to define and document processes for operator and practitioner proficiency with ai system performance

Manual factual verification: PENDING

Q04

#### FOCUSED GENERATIVE RAG: 15-QUESTION EVALUATION

In [40]:
import json
import pandas as pd
from pathlib import Path

BASE = Path("/content/rag_from_scratch")
EVAL_DIR = BASE / "evaluation"

questions_df = pd.read_csv(
    EVAL_DIR / "questions_15.csv"
)

assert len(questions_df) == 15

records = []

print("=" * 80)
print("FOCUSED GENERATIVE RAG — 15 QUESTIONS")
print("=" * 80)

for _, row in questions_df.iterrows():

    qid = str(row["question_id"])
    question = str(row["question"])

    result = focused_rag_generate(
        question=question,
        chunk_set="small",
        top_n=4,
        max_new_tokens=150
    )

    evidence = result["evidence"]

    evidence_records = [
        {
            "evidence_id": f"E{i}",
            "page": int(item["page"]),
            "chunk_id": item["chunk_id"],
            "score": float(item["score"]),
            "text": item["text"]
        }
        for i, item in enumerate(evidence, start=1)
    ]

    records.append({
        "question_id": qid,
        "question": question,
        "gold_page": int(row["gold_page"]),
        "generated_answer": result["answer"],
        "abstained": bool(result["abstained"]),
        "evidence_pages": sorted(set(
            item["page"]
            for item in evidence_records
        )),
        "gold_page_in_evidence": (
            int(row["gold_page"])
            in [
                item["page"]
                for item in evidence_records
            ]
        ),
        "evidence": evidence_records,
        "manual_answer_correct": None,
        "manual_evidence_supported": None,
        "review_notes": ""
    })

    print("\n" + "-" * 80)
    print(f"{qid}: {question}")

    print("\nGENERATED ANSWER:")
    print(result["answer"])

    print("\nABSTAINED:", result["abstained"])

    print("\nSUPPLIED EVIDENCE:")

    for item in evidence_records:
        print(
            f"[{item['evidence_id']}] "
            f"PDF page {item['page']} | "
            f"Chunk {item['chunk_id']}"
        )
        print(item["text"])

# ============================================================
# SAVE FULL RESULTS
# ============================================================

json_path = (
    EVAL_DIR / "focused_generative_rag_15.json"
)

csv_path = (
    EVAL_DIR / "focused_generative_rag_15.csv"
)

with open(json_path, "w", encoding="utf-8") as f:
    json.dump(
        records,
        f,
        indent=2,
        ensure_ascii=False
    )

csv_rows = []

for record in records:

    csv_rows.append({
        "question_id": record["question_id"],
        "question": record["question"],
        "gold_page": record["gold_page"],
        "generated_answer": record["generated_answer"],
        "abstained": record["abstained"],
        "evidence_pages": ", ".join(
            map(str, record["evidence_pages"])
        ),
        "gold_page_in_evidence": (
            record["gold_page_in_evidence"]
        ),
        "manual_answer_correct": "",
        "manual_evidence_supported": "",
        "review_notes": ""
    })

pd.DataFrame(csv_rows).to_csv(
    csv_path,
    index=False
)

# ============================================================
# SUMMARY
# ============================================================

total = len(records)

answered = sum(
    not record["abstained"]
    for record in records
)

abstained = total - answered

gold_page_available = sum(
    record["gold_page_in_evidence"]
    for record in records
)

print("\n" + "=" * 80)
print("FOCUSED GENERATIVE RAG SUMMARY")
print("=" * 80)

print("Questions:", total)
print("Answered:", answered)
print("Abstained:", abstained)
print("Gold page present in supplied evidence:", gold_page_available)
print("Manually verified answers: PENDING")

print("\nSaved:")
print(json_path)
print(csv_path)


FOCUSED GENERATIVE RAG — 15 QUESTIONS

--------------------------------------------------------------------------------
Q01: What are the four core functions of the AI RMF?

GENERATED ANSWER:
The four core functions of the AI RMF are governance, mapping, measurement, and management.

ABSTAINED: False

SUPPLIED EVIDENCE:
[E1] PDF page 25 | Chunk p25_t0
ai rmf core the ai rmf core provides outcomes and actions that enable dialogue, understanding, and activities to manage ai risks and responsibly develop trustworthy ai systems. as illus - trated in figure 5, the core is composed of four functions : govern, map, measure, and manage.
[E2] PDF page 25 | Chunk p25_t104
ai rmf core functions should be carried out in a way that reflects diverse and multidisciplinary perspectives, potentially including the views of ai actors out - side the organization
[E3] PDF page 15 | Chunk p15_t208
the ai rmf functions, described in section 5, require diverse perspectives, disciplines, professions, and exper

## FOCUSED RAG ABSTENTION TEST


In [41]:
import json
from pathlib import Path

test_cases = [
    {
        "id": "U01",
        "question": (
            "What was NVIDIA's closing stock price "
            "on January 5, 2024?"
        ),
        "expected": "abstain"
    },
    {
        "id": "U02",
        "question": (
            "Who won the 2022 FIFA World Cup?"
        ),
        "expected": "abstain"
    },
    {
        "id": "U03",
        "question": (
            "What is the population of Tokyo in 2024?"
        ),
        "expected": "abstain"
    },
    {
        "id": "A01",
        "question": (
            "What are the four core functions "
            "of the AI RMF?"
        ),
        "expected": "answer"
    }
]

results = []

for case in test_cases:

    result = focused_rag_generate(
        question=case["question"],
        chunk_set="small",
        top_n=4,
        max_new_tokens=150
    )

    results.append({
        "id": case["id"],
        "question": case["question"],
        "expected": case["expected"],
        "answer": result["answer"],
        "abstained": bool(result["abstained"]),
        "evidence_pages": [
            int(item["page"])
            for item in result["evidence"]
        ]
    })

    print("\n" + "=" * 75)
    print("TEST:", case["id"])
    print("QUESTION:", case["question"])
    print("EXPECTED:", case["expected"])

    print("\nGENERATED ANSWER:")
    print(result["answer"])

    print("\nABSTAINED:", result["abstained"])
    print(
        "EVIDENCE PAGES:",
        [
            item["page"]
            for item in result["evidence"]
        ]
    )


# ============================================================
# SAVE RESULTS
# ============================================================

output_path = Path(
    "/content/rag_from_scratch/evaluation/"
    "focused_rag_abstention_tests.json"
)

with open(
    output_path,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        results,
        f,
        indent=2,
        ensure_ascii=False
    )


# ============================================================
# SUMMARY
# ============================================================

correct_behavior = sum(
    (
        r["expected"] == "abstain"
        and r["abstained"]
    )
    or
    (
        r["expected"] == "answer"
        and not r["abstained"]
    )
    for r in results
)

print("\n" + "=" * 75)
print("ABSTENTION TEST SUMMARY")
print("=" * 75)

print("Total tests:", len(results))
print("Expected behavior observed:", correct_behavior)
print("Unexpected behavior:", len(results) - correct_behavior)
print("Saved:", output_path)



TEST: U01
QUESTION: What was NVIDIA's closing stock price on January 5, 2024?
EXPECTED: abstain

GENERATED ANSWER:
NVIDIA's closing stock price on January 5, 2024 was $16.50.

ABSTAINED: False
EVIDENCE PAGES: [3, 2, 37, 29]

TEST: U02
QUESTION: Who won the 2022 FIFA World Cup?
EXPECTED: abstain

GENERATED ANSWER:
The 2022 FIFA World Cup was won by South Africa.

ABSTAINED: False
EVIDENCE PAGES: [7, 7, 15, 14]

TEST: U03
QUESTION: What is the population of Tokyo in 2024?
EXPECTED: abstain

GENERATED ANSWER:
The population of Tokyo in 2024 is approximately 38 million.

ABSTAINED: False
EVIDENCE PAGES: [3, 11, 7, 14]

TEST: A01
QUESTION: What are the four core functions of the AI RMF?
EXPECTED: answer

GENERATED ANSWER:
The four core functions of the AI RMF are governance, mapping, measurement, and management.

ABSTAINED: False
EVIDENCE PAGES: [25, 25, 15, 8]

ABSTENTION TEST SUMMARY
Total tests: 4
Expected behavior observed: 1
Unexpected behavior: 3
Saved: /content/rag_from_scratch/eval

In [42]:
# EVIDENCE-GATED FOCUSED GENERATIVE RAG

import json
from pathlib import Path

ABSTAIN = "Insufficient information in the retrieved document."


def final_rag_generate(
    question,
    chunk_set="small",
    top_k=5,
    min_coverage=0.45,
    top_n=4,
    max_new_tokens=150
):
    """
    Combines:
    1. Hybrid retrieval
    2. Evidence relevance gate
    3. Focused evidence selection
    4. Local Qwen LLM inference

    Does not claim automatic factual verification.
    """

    # --------------------------------------------------------
    # 1. Retrieve candidate evidence
    # --------------------------------------------------------

    retrieved = hybrid_retrieve(
        query=question,
        chunk_set=chunk_set,
        top_k=top_k,
        candidate_k=30,
        rrf_k=60
    )

    # --------------------------------------------------------
    # 2. Apply evidence gate BEFORE LLM inference
    # --------------------------------------------------------

    relevance = evidence_relevance_check(
        question=question,
        retrieved=retrieved,
        min_coverage=min_coverage
    )

    if not relevance["relevant"]:

        return {
            "question": question,
            "answer": ABSTAIN,
            "abstained": True,
            "abstention_reason": "evidence_gate",
            "llm_called": False,
            "lexical_coverage": relevance["coverage"],
            "matched_terms": relevance["matched_terms"],
            "evidence": []
        }

    # --------------------------------------------------------
    # 3. Generate from focused evidence
    # --------------------------------------------------------

    result = focused_rag_generate(
        question=question,
        chunk_set=chunk_set,
        top_n=top_n,
        max_new_tokens=max_new_tokens
    )

    return {
        "question": question,
        "answer": result["answer"],
        "abstained": bool(result["abstained"]),
        "abstention_reason": (
            "llm_abstention"
            if result["abstained"]
            else None
        ),
        "llm_called": True,
        "lexical_coverage": relevance["coverage"],
        "matched_terms": relevance["matched_terms"],
        "evidence": result["evidence"]
    }


# ============================================================
# 4. TEST THE ORIGINAL FOUR CASES + Q15
# ============================================================

test_cases = [
    {
        "id": "U01",
        "question": (
            "What was NVIDIA's closing stock price "
            "on January 5, 2024?"
        ),
        "expected": "abstain"
    },
    {
        "id": "U02",
        "question": "Who won the 2022 FIFA World Cup?",
        "expected": "abstain"
    },
    {
        "id": "U03",
        "question": "What is the population of Tokyo in 2024?",
        "expected": "abstain"
    },
    {
        "id": "A01",
        "question": "What are the four core functions of the AI RMF?",
        "expected": "answer"
    },
    {
        "id": "A15",
        "question": (
            "Why is AI risk management considered "
            "a continuous process?"
        ),
        "expected": "answer"
    }
]

test_results = []

print("=" * 80)
print("EVIDENCE-GATED FOCUSED RAG — TESTS")
print("=" * 80)

for case in test_cases:

    result = final_rag_generate(
        question=case["question"],
        chunk_set="small",
        min_coverage=0.45
    )

    expected_abstention = (
        case["expected"] == "abstain"
    )

    behavior_correct = (
        result["abstained"] == expected_abstention
    )

    test_results.append({
        "id": case["id"],
        "question": case["question"],
        "expected": case["expected"],
        "answer": result["answer"],
        "abstained": result["abstained"],
        "llm_called": result["llm_called"],
        "lexical_coverage": result["lexical_coverage"],
        "abstention_reason": result["abstention_reason"],
        "expected_behavior_observed": behavior_correct
    })

    print("\n" + "-" * 80)

    print("TEST:", case["id"])
    print("Question:", case["question"])
    print("Expected:", case["expected"])
    print("Lexical coverage:", result["lexical_coverage"])
    print("LLM called:", result["llm_called"])

    print("\nAnswer:")
    print(result["answer"])

    print("\nAbstained:", result["abstained"])
    print("Reason:", result["abstention_reason"])
    print("Expected behavior observed:", behavior_correct)


# ============================================================
# 5. SAVE RESULTS
# ============================================================

EVAL_DIR = Path(
    "/content/rag_from_scratch/evaluation"
)

EVAL_DIR.mkdir(parents=True, exist_ok=True)

output_path = (
    EVAL_DIR / "final_rag_gate_tests.json"
)

with open(
    output_path,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        test_results,
        f,
        indent=2,
        ensure_ascii=False
    )


# ============================================================
# 6. SUMMARY
# ============================================================

correct = sum(
    r["expected_behavior_observed"]
    for r in test_results
)

print("\n" + "=" * 80)
print("FINAL GATE TEST SUMMARY")
print("=" * 80)

print("Total tests:", len(test_results))
print("Expected behavior observed:", correct)
print("Unexpected behavior:", len(test_results) - correct)

print("\nSaved:", output_path)

EVIDENCE-GATED FOCUSED RAG — TESTS

--------------------------------------------------------------------------------
TEST: U01
Question: What was NVIDIA's closing stock price on January 5, 2024?
Expected: abstain
Lexical coverage: 0.1667
LLM called: False

Answer:
Insufficient information in the retrieved document.

Abstained: True
Reason: evidence_gate
Expected behavior observed: True

--------------------------------------------------------------------------------
TEST: U02
Question: Who won the 2022 FIFA World Cup?
Expected: abstain
Lexical coverage: 0.3333
LLM called: False

Answer:
Insufficient information in the retrieved document.

Abstained: True
Reason: evidence_gate
Expected behavior observed: True

--------------------------------------------------------------------------------
TEST: U03
Question: What is the population of Tokyo in 2024?
Expected: abstain
Lexical coverage: 0.3333
LLM called: False

Answer:
Insufficient information in the retrieved document.

Abstained: True


## Final evaluation of the complete RAG pipeline

In [43]:
import json
import pandas as pd
from pathlib import Path

BASE = Path("/content/rag_from_scratch")
EVAL_DIR = BASE / "evaluation"
EVAL_DIR.mkdir(parents=True, exist_ok=True)

questions_df = pd.read_csv(
    EVAL_DIR / "questions_15.csv"
)

assert len(questions_df) == 15

records = []

print("=" * 75)
print("FINAL GENERATIVE RAG — 15 ANSWERABLE QUESTIONS")
print("=" * 75)

for _, row in questions_df.iterrows():

    qid = str(row["question_id"])
    question = str(row["question"])

    result = final_rag_generate(
        question=question,
        chunk_set="small",
        min_coverage=0.45,
        top_n=4,
        max_new_tokens=150
    )

    evidence = [
        {
            "evidence_id": f"E{i}",
            "page": int(item["page"]),
            "chunk_id": item["chunk_id"],
            "score": float(item["score"]),
            "text": item["text"]
        }
        for i, item in enumerate(
            result["evidence"], start=1
        )
    ]

    gold_page = int(row["gold_page"])

    record = {
        "question_id": qid,
        "question": question,
        "expected": "answer",
        "gold_page": gold_page,
        "answer": result["answer"],
        "abstained": bool(result["abstained"]),
        "llm_called": bool(result["llm_called"]),
        "lexical_coverage": float(
            result["lexical_coverage"]
        ),
        "abstention_reason": result["abstention_reason"],
        "evidence": evidence,
        "gold_page_in_evidence": any(
            item["page"] == gold_page
            for item in evidence
        ),
        "manual_answer_correct": None,
        "manual_evidence_supported": None,
        "review_notes": ""
    }

    records.append(record)

    print(f"\n{qid}: {question}")
    print("Answer:", record["answer"])
    print("Abstained:", record["abstained"])
    print(
        "Evidence pages:",
        sorted(set(
            item["page"]
            for item in evidence
        ))
    )


# ============================================================
# OUT-OF-SCOPE TEST CASES
# ============================================================

unsupported_questions = [
    (
        "U01",
        "What was NVIDIA's closing stock price "
        "on January 5, 2024?"
    ),
    (
        "U02",
        "Who won the 2022 FIFA World Cup?"
    ),
    (
        "U03",
        "What is the population of Tokyo in 2024?"
    )
]

unsupported_records = []

for qid, question in unsupported_questions:

    result = final_rag_generate(
        question=question,
        chunk_set="small",
        min_coverage=0.45,
        top_n=4,
        max_new_tokens=150
    )

    unsupported_records.append({
        "question_id": qid,
        "question": question,
        "expected": "abstain",
        "answer": result["answer"],
        "abstained": bool(result["abstained"]),
        "llm_called": bool(result["llm_called"]),
        "lexical_coverage": float(
            result["lexical_coverage"]
        ),
        "abstention_reason": result["abstention_reason"]
    })


# ============================================================
# SAVE RESULTS
# ============================================================

answer_path = EVAL_DIR / "final_generative_rag_15.json"

unsupported_path = (
    EVAL_DIR / "final_rag_unsupported_3.json"
)

csv_path = EVAL_DIR / "final_generative_rag_15.csv"

answer_path.write_text(
    json.dumps(records, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

unsupported_path.write_text(
    json.dumps(
        unsupported_records,
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8"
)

pd.DataFrame([
    {
        "question_id": r["question_id"],
        "question": r["question"],
        "answer": r["answer"],
        "abstained": r["abstained"],
        "llm_called": r["llm_called"],
        "lexical_coverage": r["lexical_coverage"],
        "gold_page_in_evidence": r["gold_page_in_evidence"],
        "evidence_pages": ", ".join(
            map(
                str,
                sorted(set(
                    item["page"]
                    for item in r["evidence"]
                ))
            )
        ),
        "manual_answer_correct": "",
        "manual_evidence_supported": "",
        "review_notes": ""
    }
    for r in records
]).to_csv(csv_path, index=False)


# ============================================================
# SUMMARY
# ============================================================

answered = sum(
    not r["abstained"]
    for r in records
)

unsupported_correct = sum(
    r["abstained"]
    for r in unsupported_records
)

gold_pages = sum(
    r["gold_page_in_evidence"]
    for r in records
)

print("\n" + "=" * 75)
print("FINAL END-TO-END EVALUATION SUMMARY")
print("=" * 75)

print("Answerable questions:", len(records))
print("Answers generated:", answered)
print("Answerable questions abstained:", len(records) - answered)
print("Gold page in supplied evidence:", gold_pages)

print("\nUnsupported questions:", len(unsupported_records))
print("Correct abstentions:", unsupported_correct)

print("\nUnsupported question details:")

for r in unsupported_records:
    print(
        r["question_id"],
        "| Abstained:", r["abstained"],
        "| LLM called:", r["llm_called"],
        "| Coverage:", r["lexical_coverage"]
    )

print("\nManual factual verification: PENDING")

print("\nSaved files:")
print(answer_path)
print(csv_path)
print(unsupported_path)


FINAL GENERATIVE RAG — 15 ANSWERABLE QUESTIONS

Q01: What are the four core functions of the AI RMF?
Answer: The four core functions of the AI RMF are governance, mapping, measurement, and management.
Abstained: False
Evidence pages: [8, 15, 25]

Q02: What is the purpose of the GOVERN function?
Answer: The GOVERN function aims to provide organizations with clarity about AI roles and responsibilities, create mechanisms for decision-making processes to be more transparent, and foster a risk-driven culture within an organization.
Abstained: False
Evidence pages: [27, 30, 46]

Q03: What does the MAP function establish about an AI system?
Answer: The MAP function establishes that risks are framed by the context of an AI system.
Abstained: False
Evidence pages: [29, 46]

Q04: What is the purpose of the MEASURE function?
Answer: The purpose of the MEASURE function is to provide a basis for measuring and managing AI risk and impact through quantitative, qualitative, or mixed-methods analysis, 

#### Record the answer-quality review


In [44]:
# RECORD PRELIMINARY ANSWER QUALITY REVIEW

import json
import pandas as pd
from pathlib import Path

EVAL_DIR = Path("/content/rag_from_scratch/evaluation")

input_path = EVAL_DIR / "final_generative_rag_15.json"

with open(input_path, encoding="utf-8") as f:
    records = json.load(f)

# Qualitative review of the displayed answers.
# "Adequate" does not imply independent factual verification.

review = {
    "Q01": ("adequate", "All four core functions identified."),
    "Q02": ("adequate", "Describes GOVERN responsibilities and risk culture."),
    "Q03": ("adequate", "Correctly explains context for framing risks."),
    "Q04": ("adequate", "Identifies the main measurement methods and activities."),
    "Q05": ("partial", "Missing risk prioritization and response strategies."),
    "Q06": ("partial", "Omits resilience, explainability and interpretability."),
    "Q07": ("partial", "Reliability under expected conditions and over time is incomplete."),
    "Q08": ("adequate", "Describes NIST safety practices."),
    "Q09": ("partial", "Addresses resilience but insufficiently explains security."),
    "Q10": ("adequate", "Correctly distinguishes explainability from interpretability."),
    "Q11": ("adequate", "Describes relevant privacy principles."),
    "Q12": ("adequate", "Explains harmful bias in relation to fairness."),
    "Q13": ("adequate", "Explains the purpose of AI RMF profiles."),
    "Q14": ("partial", "Transparency is explained better than accountability."),
    "Q15": ("adequate", "Explains ongoing monitoring across the AI lifecycle.")
}

assert len(records) == 15
assert {r["question_id"] for r in records} == set(review)

review_records = []

for record in records:
    qid = record["question_id"]
    status, notes = review[qid]

    review_records.append({
        "question_id": qid,
        "question": record["question"],
        "generated_answer": record["answer"],
        "review_status": status,
        "review_notes": notes,
        "review_type": "preliminary_qualitative",
        "independently_verified": False
    })

df = pd.DataFrame(review_records)

csv_path = EVAL_DIR / "final_answer_quality_review.csv"
json_path = EVAL_DIR / "final_answer_quality_review.json"

df.to_csv(csv_path, index=False)

json_path.write_text(
    json.dumps(review_records, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("=" * 75)
print("PRELIMINARY ANSWER QUALITY REVIEW")
print("=" * 75)

print(df[[
    "question_id",
    "review_status",
    "review_notes"
]].to_string(index=False))

print("\nReview counts:")
print(df["review_status"].value_counts().to_string())

print("\nIndependently verified: No")
print("Claim-level citation validation: Not implemented")

print("\nSaved:")
print(csv_path)
print(json_path)


PRELIMINARY ANSWER QUALITY REVIEW
question_id review_status                                                       review_notes
        Q01      adequate                                All four core functions identified.
        Q02      adequate                Describes GOVERN responsibilities and risk culture.
        Q03      adequate                      Correctly explains context for framing risks.
        Q04      adequate            Identifies the main measurement methods and activities.
        Q05       partial               Missing risk prioritization and response strategies.
        Q06       partial             Omits resilience, explainability and interpretability.
        Q07       partial Reliability under expected conditions and over time is incomplete.
        Q08      adequate                                   Describes NIST safety practices.
        Q09       partial         Addresses resilience but insufficiently explains security.
        Q10      adequate      Corre

#### HARD NEGATIVE / UNSUPPORTED QUESTION TESTS

In [45]:
import json
from pathlib import Path

BASE = Path("/content/rag_from_scratch")
EVAL_DIR = BASE / "evaluation"

hard_negative_questions = [
    {
        "id": "H01",
        "question": (
            "What exact annual budget in US dollars does "
            "the NIST AI RMF assign to the GOVERN function?"
        )
    },
    {
        "id": "H02",
        "question": (
            "What exact percentage improvement in AI model "
            "accuracy does the MEASURE function guarantee?"
        )
    },
    {
        "id": "H03",
        "question": (
            "What exact numerical fairness score must every "
            "AI system achieve under the NIST AI RMF?"
        )
    },
    {
        "id": "H04",
        "question": (
            "What specific fine amount in US dollars does "
            "the AI RMF impose when organizations fail to "
            "manage AI risks?"
        )
    }
]

results = []

print("=" * 80)
print("HARD NEGATIVE EVALUATION")
print("=" * 80)

for case in hard_negative_questions:

    result = final_rag_generate(
        question=case["question"],
        chunk_set="small",
        top_k=5,
        min_coverage=0.45,
        top_n=4,
        max_new_tokens=150
    )

    record = {
        "question_id": case["id"],
        "question": case["question"],
        "expected": "abstain",
        "answer": result["answer"],
        "abstained": bool(result["abstained"]),
        "llm_called": bool(result["llm_called"]),
        "lexical_coverage": float(
            result["lexical_coverage"]
        ),
        "abstention_reason": result["abstention_reason"],
        "evidence": [
            {
                "page": int(item["page"]),
                "chunk_id": item["chunk_id"],
                "text": item["text"]
            }
            for item in result["evidence"]
        ]
    }

    results.append(record)

    print("\n" + "-" * 80)
    print("Test:", case["id"])
    print("Question:", case["question"])
    print("Lexical coverage:", record["lexical_coverage"])
    print("LLM called:", record["llm_called"])

    print("\nAnswer:")
    print(record["answer"])

    print("\nAbstained:", record["abstained"])
    print("Reason:", record["abstention_reason"])

    if record["evidence"]:
        print("\nEvidence pages:", [
            item["page"]
            for item in record["evidence"]
        ])


# ------------------------------------------------------------
# Save results
# ------------------------------------------------------------

output_path = EVAL_DIR / "hard_negative_evaluation.json"

output_path.write_text(
    json.dumps(
        results,
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8"
)

correct_abstentions = sum(
    record["abstained"]
    for record in results
)

print("\n" + "=" * 80)
print("HARD NEGATIVE EVALUATION SUMMARY")
print("=" * 80)

print("Questions:", len(results))
print("Correct abstentions:", correct_abstentions)
print(
    "Unsupported answers generated:",
    len(results) - correct_abstentions
)

print("\nSaved:", output_path)

HARD NEGATIVE EVALUATION

--------------------------------------------------------------------------------
Test: H01
Question: What exact annual budget in US dollars does the NIST AI RMF assign to the GOVERN function?
Lexical coverage: 0.4444
LLM called: False

Answer:
Insufficient information in the retrieved document.

Abstained: True
Reason: evidence_gate

--------------------------------------------------------------------------------
Test: H02
Question: What exact percentage improvement in AI model accuracy does the MEASURE function guarantee?
Lexical coverage: 0.25
LLM called: False

Answer:
Insufficient information in the retrieved document.

Abstained: True
Reason: evidence_gate

--------------------------------------------------------------------------------
Test: H03
Question: What exact numerical fairness score must every AI system achieve under the NIST AI RMF?
Lexical coverage: 0.2727
LLM called: False

Answer:
Insufficient information in the retrieved document.

Abstained

In [46]:
# NUMERICAL CLAIM VERIFICATION

import re
import json
from pathlib import Path

ABSTAIN = "Insufficient information in the retrieved document."


def extract_numerical_claims(text):
    """
    Extract explicit numerical claims from generated text.

    Handles:
    - Monetary amounts: $50,000
    - Percentages: 25%
    - Other numerical values: 3.5, 2024, 1000

    This is a conservative lexical check,
    not numerical reasoning.
    """

    pattern = r"(?<!\w)\$?\d[\d,]*(?:\.\d+)?%?"

    return re.findall(pattern, text)


def normalize_number(value):
    return value.replace(",", "").strip()


def verify_generated_numbers(answer, evidence):
    """
    Verify whether numbers in the generated answer
    appear in the evidence supplied to the LLM.
    """

    answer_numbers = extract_numerical_claims(answer)

    evidence_text = " ".join(
        item["text"]
        for item in evidence
    )

    evidence_numbers = {
        normalize_number(number)
        for number in extract_numerical_claims(
            evidence_text
        )
    }

    unsupported = [
        number
        for number in answer_numbers
        if normalize_number(number)
        not in evidence_numbers
    ]

    return {
        "passed": len(unsupported) == 0,
        "answer_numbers": answer_numbers,
        "unsupported_numbers": unsupported
    }


def final_rag_generate_v2(
    question,
    chunk_set="small",
    min_coverage=0.45
):

    # Preserve the previous pipeline.
    result = final_rag_generate(
        question=question,
        chunk_set=chunk_set,
        min_coverage=min_coverage,
        top_n=4,
        max_new_tokens=150
    )

    verification = verify_generated_numbers(
        result["answer"],
        result["evidence"]
    )

    result["number_verification"] = verification

    if (
        not result["abstained"]
        and not verification["passed"]
    ):

        result["raw_unverified_answer"] = result["answer"]

        result["answer"] = ABSTAIN
        result["abstained"] = True
        result["abstention_reason"] = (
            "unsupported_numerical_claim"
        )

    return result


# ============================================================
# TEST: ORIGINAL HARD NEGATIVES + ANSWERABLE QUESTIONS
# ============================================================

test_cases = [
    {
        "id": "H01",
        "question": (
            "What exact annual budget in US dollars "
            "does the NIST AI RMF assign to the GOVERN function?"
        ),
        "expected": "abstain"
    },
    {
        "id": "H02",
        "question": (
            "What exact percentage improvement in AI model "
            "accuracy does the MEASURE function guarantee?"
        ),
        "expected": "abstain"
    },
    {
        "id": "H03",
        "question": (
            "What exact numerical fairness score must every "
            "AI system achieve under the NIST AI RMF?"
        ),
        "expected": "abstain"
    },
    {
        "id": "H04",
        "question": (
            "What specific fine amount in US dollars does "
            "the AI RMF impose when organizations fail to "
            "manage AI risks?"
        ),
        "expected": "abstain"
    },
    {
        "id": "A01",
        "question": (
            "What are the four core functions of the AI RMF?"
        ),
        "expected": "answer"
    },
    {
        "id": "A15",
        "question": (
            "Why is AI risk management considered "
            "a continuous process?"
        ),
        "expected": "answer"
    }
]

test_results = []

print("=" * 80)
print("NUMERICAL CLAIM VERIFICATION TEST")
print("=" * 80)

for case in test_cases:

    result = final_rag_generate_v2(
        case["question"]
    )

    expected_abstention = (
        case["expected"] == "abstain"
    )

    passed = (
        result["abstained"] == expected_abstention
    )

    record = {
        "id": case["id"],
        "question": case["question"],
        "expected": case["expected"],
        "answer": result["answer"],
        "abstained": result["abstained"],
        "abstention_reason": result["abstention_reason"],
        "number_verification": result["number_verification"],
        "passed": passed
    }

    test_results.append(record)

    print("\n" + "-" * 80)
    print("Test:", case["id"])
    print("Expected:", case["expected"])

    print("\nFinal answer:")
    print(result["answer"])

    print("\nNumerical claims:",
          result["number_verification"]["answer_numbers"])

    print("Unsupported numbers:",
          result["number_verification"]["unsupported_numbers"])

    print("Abstained:", result["abstained"])
    print("Reason:", result["abstention_reason"])
    print("Expected behavior observed:", passed)


# ============================================================
# SAVE RESULTS
# ============================================================

EVAL_DIR = Path(
    "/content/rag_from_scratch/evaluation"
)

output_path = (
    EVAL_DIR / "numerical_claim_verification_tests.json"
)

output_path.write_text(
    json.dumps(
        test_results,
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8"
)

correct = sum(
    item["passed"]
    for item in test_results
)

print("\n" + "=" * 80)
print("STEP 38 SUMMARY")
print("=" * 80)

print("Total tests:", len(test_results))
print("Expected behavior observed:", correct)
print("Unexpected behavior:", len(test_results) - correct)

print("\nSaved:", output_path)

NUMERICAL CLAIM VERIFICATION TEST

--------------------------------------------------------------------------------
Test: H01
Expected: abstain

Final answer:
Insufficient information in the retrieved document.

Numerical claims: []
Unsupported numbers: []
Abstained: True
Reason: evidence_gate
Expected behavior observed: True

--------------------------------------------------------------------------------
Test: H02
Expected: abstain

Final answer:
Insufficient information in the retrieved document.

Numerical claims: []
Unsupported numbers: []
Abstained: True
Reason: evidence_gate
Expected behavior observed: True

--------------------------------------------------------------------------------
Test: H03
Expected: abstain

Final answer:
Insufficient information in the retrieved document.

Numerical claims: []
Unsupported numbers: []
Abstained: True
Reason: evidence_gate
Expected behavior observed: True

--------------------------------------------------------------------------------
Te

In [47]:
# FINAL RAG REGRESSION EVALUATION

import json
import pandas as pd
from pathlib import Path

BASE = Path("/content/rag_from_scratch")
EVAL = BASE / "evaluation"
RESULTS = BASE / "results"

EVAL.mkdir(parents=True, exist_ok=True)
RESULTS.mkdir(parents=True, exist_ok=True)

questions = pd.read_csv(EVAL / "questions_15.csv")

assert len(questions) == 15

unsupported = [
    ("U01", "What was NVIDIA's closing stock price on January 5, 2024?"),
    ("U02", "Who won the 2022 FIFA World Cup?"),
    ("U03", "What is the population of Tokyo in 2024?"),
    ("H01", "What exact annual budget in US dollars does the NIST AI RMF assign to the GOVERN function?"),
    ("H02", "What exact percentage improvement in AI model accuracy does the MEASURE function guarantee?"),
    ("H03", "What exact numerical fairness score must every AI system achieve under the NIST AI RMF?"),
    ("H04", "What specific fine amount in US dollars does the AI RMF impose when organizations fail to manage AI risks?")
]

records = []

# ------------------------------------------------------------
# 1. Answerable questions
# ------------------------------------------------------------

print("=" * 75)
print("FINAL PIPELINE — ANSWERABLE QUESTIONS")
print("=" * 75)

for _, row in questions.iterrows():

    qid = str(row["question_id"])
    question = str(row["question"])

    result = final_rag_generate_v2(question)

    evidence = result["evidence"]

    record = {
        "question_id": qid,
        "question": question,
        "expected": "answer",
        "answer": result["answer"],
        "abstained": bool(result["abstained"]),
        "abstention_reason": result["abstention_reason"],
        "gold_page": int(row["gold_page"]),
        "gold_page_in_evidence": any(
            int(item["page"]) == int(row["gold_page"])
            for item in evidence
        ),
        "evidence_pages": sorted(set(
            int(item["page"]) for item in evidence
        )),
        "numerical_check_passed": result[
            "number_verification"
        ]["passed"],
        "manual_factual_verification": "pending"
    }

    records.append(record)

    print(
        f"{qid} | "
        f"{'ABSTAINED' if record['abstained'] else 'ANSWERED'} "
        f"| Gold page in evidence: "
        f"{record['gold_page_in_evidence']}"
    )


# ------------------------------------------------------------
# 2. Unsupported questions
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("FINAL PIPELINE — UNSUPPORTED QUESTIONS")
print("=" * 75)

for qid, question in unsupported:

    result = final_rag_generate_v2(question)

    record = {
        "question_id": qid,
        "question": question,
        "expected": "abstain",
        "answer": result["answer"],
        "abstained": bool(result["abstained"]),
        "abstention_reason": result["abstention_reason"],
        "numerical_check_passed": result[
            "number_verification"
        ]["passed"]
    }

    records.append(record)

    print(
        f"{qid} | "
        f"{'ABSTAINED' if record['abstained'] else 'UNSUPPORTED ANSWER'}"
        f" | Reason: {record['abstention_reason']}"
    )


# ------------------------------------------------------------
# 3. Calculate summary
# ------------------------------------------------------------

answerable_records = [
    r for r in records
    if r["expected"] == "answer"
]

unsupported_records = [
    r for r in records
    if r["expected"] == "abstain"
]

answered = sum(
    not r["abstained"]
    for r in answerable_records
)

correct_abstentions = sum(
    r["abstained"]
    for r in unsupported_records
)

gold_pages = sum(
    r["gold_page_in_evidence"]
    for r in answerable_records
)

summary = {
    "model": "Qwen/Qwen2.5-0.5B-Instruct",
    "retrieval": "Hybrid MiniLM + BM25 + RRF",
    "generation": "Focused evidence",
    "safeguards": [
        "Lexical evidence gate",
        "Numerical-presence verification"
    ],
    "answerable_questions": len(answerable_records),
    "answers_generated": answered,
    "answerable_abstentions": (
        len(answerable_records) - answered
    ),
    "gold_page_in_evidence": gold_pages,
    "unsupported_questions": len(unsupported_records),
    "correct_unsupported_abstentions": correct_abstentions,
    "unsupported_answers_generated": (
        len(unsupported_records) - correct_abstentions
    ),
    "manual_answer_accuracy": None,
    "claim_level_citation_verification": False
}


# ------------------------------------------------------------
# 4. Save outputs
# ------------------------------------------------------------

detail_path = EVAL / "final_rag_v2_all_22.json"
summary_path = RESULTS / "final_rag_v2_summary.json"

detail_path.write_text(
    json.dumps(records, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

summary_path.write_text(
    json.dumps(summary, indent=2, ensure_ascii=False),
    encoding="utf-8"
)


# ------------------------------------------------------------
# 5. Display results
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("FINAL PIPELINE SUMMARY")
print("=" * 75)

print("Answerable questions:", len(answerable_records))
print("Answers generated:", answered)
print("Answerable abstentions:",
      len(answerable_records) - answered)

print("\nGold page in supplied evidence:", gold_pages)

print("\nUnsupported questions:", len(unsupported_records))
print("Correct abstentions:", correct_abstentions)
print(
    "Unsupported answers generated:",
    len(unsupported_records) - correct_abstentions
)

print("\nManual answer accuracy: NOT YET MEASURED")
print("Claim-level factual verification: NOT IMPLEMENTED")

print("\nSaved:")
print(detail_path)
print(summary_path)


FINAL PIPELINE — ANSWERABLE QUESTIONS
Q01 | ANSWERED | Gold page in evidence: True
Q02 | ANSWERED | Gold page in evidence: True
Q03 | ANSWERED | Gold page in evidence: True
Q04 | ANSWERED | Gold page in evidence: True
Q05 | ANSWERED | Gold page in evidence: True
Q06 | ANSWERED | Gold page in evidence: True
Q07 | ANSWERED | Gold page in evidence: True
Q08 | ANSWERED | Gold page in evidence: True
Q09 | ANSWERED | Gold page in evidence: True
Q10 | ANSWERED | Gold page in evidence: True
Q11 | ANSWERED | Gold page in evidence: True
Q12 | ANSWERED | Gold page in evidence: True
Q13 | ANSWERED | Gold page in evidence: True
Q14 | ANSWERED | Gold page in evidence: True
Q15 | ANSWERED | Gold page in evidence: True

FINAL PIPELINE — UNSUPPORTED QUESTIONS
U01 | ABSTAINED | Reason: evidence_gate
U02 | ABSTAINED | Reason: evidence_gate
U03 | ABSTAINED | Reason: evidence_gate
H01 | ABSTAINED | Reason: evidence_gate
H02 | ABSTAINED | Reason: evidence_gate
H03 | ABSTAINED | Reason: evidence_gate
H04 | A

## Test Groq inference with existing RAG retrieval

In [49]:
%pip -q install groq

In [52]:


import time
import re
from groq import Groq

# API key is already defined in an earlier Colab cell:
# api_key = "..."

assert isinstance(api_key, str) and api_key.strip(), (
    "api_key is missing."
)

GROQ_MODEL = "openai/gpt-oss-20b"

client = Groq(api_key=api_key.strip())

# ------------------------------------------------------------
# 1. Test question
# ------------------------------------------------------------

question = (
    "What is the purpose of the MEASURE function?"
)

# ------------------------------------------------------------
# 2. Reuse our existing evidence selection
# ------------------------------------------------------------

evidence = select_complete_evidence(
    question=question,
    chunk_set="small",
    candidate_k=15,
    top_n=3,
    min_words=12
)

assert evidence, "No evidence retrieved."

context = "\n\n".join(
    f"[E{i}] PDF page {item['page']}\n"
    f"{item['text']}"
    for i, item in enumerate(evidence, start=1)
)

# ------------------------------------------------------------
# 3. Grounded prompt
# ------------------------------------------------------------

system_prompt = """
You are a document-grounded question-answering assistant.

Rules:
1. Use ONLY the supplied evidence.
2. Answer the question directly in 1-3 sentences.
3. Do not invent facts, figures, or references.
4. Cite the supporting evidence using [E1], [E2], etc.
5. Only cite evidence that directly supports the answer.
6. If the evidence does not answer the question, reply exactly:
Insufficient information in the retrieved document.
7. Do not use outside knowledge.
""".strip()

user_prompt = f"""
Question:
{question}

Retrieved evidence:
{context}

Provide a concise, evidence-grounded answer.
""".strip()

# ------------------------------------------------------------
# 4. Groq API inference
# ------------------------------------------------------------

start = time.perf_counter()

completion = client.chat.completions.create(
    model=GROQ_MODEL,
    messages=[
        {
            "role": "system",
            "content": system_prompt
        },
        {
            "role": "user",
            "content": user_prompt
        }
    ],
    reasoning_effort="low",
    temperature=0,
    max_completion_tokens=350,
)

elapsed = time.perf_counter() - start

answer = (
    completion.choices[0].message.content or ""
).strip()

# ------------------------------------------------------------
# 5. Citation validation
# ------------------------------------------------------------

cited_ids = re.findall(
    r"\[E(\d+)\]",
    answer
)

invalid_ids = [
    number
    for number in cited_ids
    if not (1 <= int(number) <= len(evidence))
]

abstained = (
    answer.rstrip(".").lower()
    == "insufficient information in the retrieved document"
)

citation_format_pass = (
    abstained
    or (
        len(cited_ids) > 0
        and len(invalid_ids) == 0
    )
)

# ------------------------------------------------------------
# 6. Usage statistics
# ------------------------------------------------------------

usage = completion.usage

print("=" * 75)
print("GROQ GPT-OSS-20B — RAG TEST")
print("=" * 75)

print("Model:", completion.model)
print("Question:", question)

print("\nGENERATED ANSWER:")
print(answer)

print("\nRETRIEVED EVIDENCE:")

for i, item in enumerate(evidence, start=1):
    print(
        f"[E{i}] PDF page {item['page']} "
        f"| Chunk {item['chunk_id']}"
    )
    print(item["text"])

print("\nCITATION CHECK:")
print("Cited IDs:", cited_ids)
print("Invalid IDs:", invalid_ids)
print("Citation format passed:", citation_format_pass)
print("Abstained:", abstained)

print("\nAPI USAGE:")
print("Prompt tokens:", usage.prompt_tokens)
print("Completion tokens:", usage.completion_tokens)
print("Total tokens:", usage.total_tokens)
print("Latency (seconds):", round(elapsed, 3))
print("Finish reason:", completion.choices[0].finish_reason)


GROQ GPT-OSS-20B — RAG TEST
Model: openai/gpt-oss-20b
Question: What is the purpose of the MEASURE function?

GENERATED ANSWER:
The MEASURE function is used to analyze, assess, benchmark, and monitor AI risk and related impacts using quantitative, qualitative, or mixed‑method tools, techniques, and methodologies, producing outcomes that feed into the manage function for risk monitoring and response [E2][E3].

RETRIEVED EVIDENCE:
[E1] PDF page 30 | Chunk p30_t0
outcomes in the map function are the basis for the measure and manage functions.
[E2] PDF page 33 | Chunk p33_t0
3 measure the measure function employs quantitative, qualitative, or mixed - method tools, tech - niques, and methodologies to analyze, assess, benchmark, and monitor ai risk and related impacts.
[E3] PDF page 33 | Chunk p33_t416
measurement outcomes will be utilized in the manage function to assist risk monitoring and response efforts. it is in - cumbent on framework users to continue applying the measure function to 

In [53]:
# GROQ GPT-OSS-20B: 15-QUESTION EVALUATION

import json
import re
import time
import pandas as pd
from pathlib import Path
from groq import RateLimitError

BASE = Path("/content/rag_from_scratch")
EVAL = BASE / "evaluation"

questions = pd.read_csv(EVAL / "questions_15.csv")
assert len(questions) == 15

ABSTAIN = "Insufficient information in the retrieved document."

groq_results = []

print("=" * 75)
print("GROQ GPT-OSS-20B — 15-QUESTION EVALUATION")
print("=" * 75)

for _, row in questions.iterrows():

    qid = str(row["question_id"])
    question = str(row["question"])

    evidence = select_complete_evidence(
        question=question,
        chunk_set="small",
        candidate_k=15,
        top_n=3,
        min_words=12
    )

    context = "\n\n".join(
        f"[E{i}] PDF page {item['page']}\n{item['text']}"
        for i, item in enumerate(evidence, 1)
    )

    messages = [
        {
            "role": "system",
            "content": (
                "You answer questions using only the supplied "
                "NIST AI RMF source passages. "
                "Answer directly in one to three sentences. "
                "Include every important part of the answer. "
                "Cite supporting passages with [E1], [E2], etc. "
                "Never invent facts or citations. "
                "If evidence is insufficient, reply exactly: "
                + ABSTAIN
            )
        },
        {
            "role": "user",
            "content": (
                f"Question:\n{question}\n\n"
                f"Evidence:\n{context}\n\n"
                "Provide a concise, fully supported answer."
            )
        }
    ]

    # Retry briefly if free-tier rate limit is reached.
    response = None

    for attempt in range(4):
        try:
            response = client.chat.completions.create(
                model="openai/gpt-oss-20b",
                messages=messages,
                reasoning_effort="low",
                temperature=0,
                max_completion_tokens=350
            )
            break

        except RateLimitError:
            if attempt == 3:
                raise
            print("Rate limit reached; retrying after cooldown.")
            time.sleep(30)

    answer = (
        response.choices[0].message.content or ""
    ).strip()

    citations = re.findall(r"\[E(\d+)\]", answer)

    valid_citations = (
        len(citations) > 0
        and all(
            1 <= int(n) <= len(evidence)
            for n in citations
        )
    )

    abstained = (
        answer.rstrip(".").lower()
        == ABSTAIN.rstrip(".").lower()
    )

    cited_pages = sorted(set(
        int(evidence[int(n) - 1]["page"])
        for n in citations
        if 1 <= int(n) <= len(evidence)
    ))

    groq_results.append({
        "question_id": qid,
        "question": question,
        "gold_page": int(row["gold_page"]),
        "answer": answer,
        "abstained": abstained,
        "citation_format_pass": (
            abstained or valid_citations
        ),
        "cited_pages": cited_pages,
        "evidence": [
            {
                "id": f"E{i}",
                "page": int(item["page"]),
                "chunk_id": item["chunk_id"],
                "text": item["text"]
            }
            for i, item in enumerate(evidence, 1)
        ],
        "prompt_tokens": response.usage.prompt_tokens,
        "completion_tokens": response.usage.completion_tokens,
        "total_tokens": response.usage.total_tokens,
        "finish_reason": response.choices[0].finish_reason,
        "manual_answer_review": "pending"
    })

    print("\n" + "-" * 75)
    print(qid, "-", question)
    print("\nANSWER:")
    print(answer)
    print("Citation format valid:", abstained or valid_citations)
    print("Cited PDF pages:", cited_pages)
    print("Tokens:", response.usage.total_tokens)

    # Modest pacing for free-tier requests.
    time.sleep(3)


# ============================================================
# SAVE RESULTS
# ============================================================

output = EVAL / "groq_gpt_oss_20b_15.json"

output.write_text(
    json.dumps(groq_results, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

answered = sum(not r["abstained"] for r in groq_results)
citation_passed = sum(
    r["citation_format_pass"] for r in groq_results
)

total_tokens = sum(r["total_tokens"] for r in groq_results)

print("\n" + "=" * 75)
print("GROQ EVALUATION SUMMARY")
print("=" * 75)

print("Questions:", len(groq_results))
print("Answered:", answered)
print("Abstained:", len(groq_results) - answered)
print("Citation format passed:", citation_passed)
print("Total API tokens:", total_tokens)
print("Manual factual accuracy: PENDING")
print("Saved:", output)


GROQ GPT-OSS-20B — 15-QUESTION EVALUATION

---------------------------------------------------------------------------
Q01 - What are the four core functions of the AI RMF?

ANSWER:
The four core functions of the AI RMF are **govern, map, measure, and manage**【E1】.
Citation format valid: False
Cited PDF pages: []
Tokens: 360

---------------------------------------------------------------------------
Q02 - What is the purpose of the GOVERN function?

ANSWER:
The GOVERN function clarifies and defines the roles and responsibilities of humans in AI team configurations and those overseeing AI performance, and establishes mechanisms to make decision‑making processes explicit to counter systemic biases. It is a cross‑cutting function infused throughout AI risk management, enabling the other functions and ensuring compliance and evaluation are integrated into all stages of the process. [E1] [E3]
Citation format valid: True
Cited PDF pages: [27, 46]
Tokens: 433

-------------------------------